# V21 — RealMLP dan TabM-PWL, memakai acuan Dani

**Kaggle: GPU T4 dianjurkan, lalu Run All.** Input cukup data kompetisi + paket Dani (`prediksi_oof_test.npz`, `prediksi_meta.json`). Cache v7/v14/v20 tidak diperlukan. Ini notebook mandiri; notebook lama tidak diubah.

Tiga kandidat baru:

1. `realmlp_eui`: implementasi resmi RealMLP-TD, target EUI mentah.
2. `realmlp_electric_area`: RealMLP-TD, target listrik/luas; prediksi dikalikan luas sebelum penilaian.
3. `tabm_pwl_eui`: TabM resmi dengan Piecewise Linear Embeddings versi B, target EUI mentah.

Model lama tidak dilatih ulang. Keenam job screening = tiga kandidat × dua outer fold. Setiap job mencakup pemilihan epoch pada inner holdout subzone dan refit outer-train, sehingga ada dua tahap training per job. Pilihan model, alpha, dan mode gas/EUI dikunci sesudah screening. Hanya komponen pilihan yang dilanjutkan ke tiga fold konfirmasi.

**Dependensi:** `pytabkit==1.7.3`, `tabm==0.0.3`, `rtdl_num_embeddings==0.0.12`. Paket yang belum tersedia dicoba dari wheel Input, lalu PyPI bila diizinkan konfigurasi. Internet Kaggle perlu aktif bila wheel dan dependensinya belum ada. Versi Torch/NumPy/pandas/scikit-learn yang tersedia dipertahankan melalui pip constraints. Modul yang gagal dimuat dilewati dengan alasan tertulis.

**Evaluasi:** model baru memakai fold subzone Dani dan inner holdout terpisah. Blend dengan S28/v17 tetap eksploratif karena acuan tersebut sudah memiliki koreksi berbasis OOF; ini bukan nested CV penuh seluruh pipeline. Gain tiga fold konfirmasi dilaporkan terpisah. Full CV lima fold juga mencakup bagian yang dipakai memilih resep.

Public terbaik historis: v17 **0,25176**. V20 pernah menurunkan CV namun public menjadi **0,25192**. Tidak ada jaminan model baru mencapai 0,24.

## Audit `punya_teman.ipynb` sebelum melanjutkan

Seluruh 25 sel dan output tersimpan dibaca. Tabel ini menyalin output sel 21 teman dan output v17 (`barusannyoba.ipynb`); bukan hasil eksekusi v21.

| Target | Teman | Acuan v17 |
|---|---:|---:|
| Gas NRMSE | 0,25124 | 0,250231 |
| Listrik NRMSE | 0,32038 | 0,319368 |
| EUI NRMSE | 0,66136 | 0,660092 |
| Clean NRMSE | 0,00004 | sekitar 0 |
| Adopsi NRMSE | 0,05653 | 0,117813 |
| CV gabungan | **0,25791** | **0,269501** |

**Skema validasi berbeda.** Teman memakai `StratifiedKFold(n_splits=7)` dengan strata zona, bukan split yang memisahkan zona/subzone. Lookup adopsi dari subzone training dapat dipakai pada validasi. Mekanisme lookup relevan untuk subzone test yang sudah dikenal, tetapi skornya berbeda makna dengan evaluasi subzone baru. Selisih adopsi sendiri menjelaskan sekitar 0,01226 pada skor gabungan. Gas, listrik, dan EUI teman masih sedikit lebih tinggi daripada acuan; CV tersebut belum membuktikan model energi lebih baik. Skor public teman tidak tercatat di notebook.

**Pemilihan bobot ikut memakai validasi yang dinilai.** Sel 17 mengoptimalkan blend EUI, gas, serta rekonsiliasi per bin luas memakai label validasi. Sel 19 melakukan hal serupa pada listrik. Stacking stage 2 juga memakai OOF stage 1 global, belum dibangun ulang di dalam setiap outer fold stage 2. Jadi hasil akhir belum merupakan penilaian holdout yang sepenuhnya terpisah dari proses fitting.

Representasi listrik per luas teman mendukung eksperimen `realmlp_electric_area` di sini. Rasio luas/volume/unit sebagian besar sudah ada dalam S7_density. Fitur termal dan SVD teman belum memiliki ablation yang menunjukkan gain terpisah; v21 mempertahankan fitur acuan agar perubahan model bisa dibaca. Clean tetap memakai rumus deterministik acuan. Validasi v21 tetap memakai fold subzone Dani, pemilihan epoch pada inner holdout, dan resep blend dikunci sebelum tiga fold konfirmasi. Lookup hanya diterapkan pada prediksi test.

In [1]:
#tambah
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import random
import shutil
import subprocess
import sys
import threading
import time
import traceback
import zipfile
from contextlib import contextmanager
from datetime import datetime
from itertools import combinations
from pathlib import Path
from zoneinfo import ZoneInfo

N_THREADS = min(4, os.cpu_count() or 1)
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(N_THREADS)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import pandas as pd
import joblib
import torch
from IPython.display import FileLink, display

S28_PRED_PATH = None
DATA_DIR_OVERRIDE = None
REUSE_RUN_DIR = None
RUN_TRAINING = True
AUTO_COMPLETE = True
REQUIRE_GPU = True
INSTALL_MISSING = True
ALLOW_PYPI = True
RUN_BACKUP = True
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
SEED = 42
TARGETS = ["target_macro_gas_kwh", "target_macro_elec_kwh", "target_eui_gas_kwh_per_sqm",
           "target_clean_building_ratio", "target_electric_heating_adoption"]
SIGMA = np.array([151750.04, 41725.37, 40.65, 12.49, 7.18], dtype=np.float64)
CONFIG = {
    "revision": "v21_realmlp_tabm_pwl_20261019_1", "seed": SEED,
    "screen_folds": [0, 1], "blend_alphas": [0.05, 0.10, 0.25, 0.50],
    "screen_min_gain": 0.00010, "confirm_min_gain": 0.00010, "full_min_gain": 0.00010,
    "full_min_improved_folds": 4, "inner_group_fraction": 0.15,
    "encoding_folds": 3, "zone_smoothing": 30.0,
    "realmlp": {"n_epochs": 256, "batch_size": 1024, "predict_batch_size": 4096, "query_chunk_rows": 65536,
                "patience": 16, "verbosity": 2},
    "tabm": {"epochs": 128, "patience": 16, "batch_size": 512, "predict_batch_size": 2048,
             "k": 16, "n_blocks": 2, "d_block": 256, "dropout": 0.1,
             "n_bins": 32, "d_embedding": 12, "lr": 0.002, "weight_decay": 0.0003,
             "max_seconds_per_phase": None},
    "evaluation": "new_neural_group_oof_with_cached_S28_v17_reference_exploratory",
}
SPECS = {
    "realmlp_eui": {"kind": "realmlp", "target": 2, "transform": "identity"},
    "realmlp_electric_area": {"kind": "realmlp", "target": 1, "transform": "divide_by_area"},
    "tabm_pwl_eui": {"kind": "tabm", "target": 2, "transform": "identity"},
}
POPULAR12 = ["bld_dim_01", "bld_dim_02", "bld_dim_04", "bld_dim_06", "bld_dim_07", "bld_dim_08",
             "bld_dim_12", "bld_dim_15", "bld_dim_20", "bld_dim_32", "socio_demog_082", "env_climate_05"]
TE_NAMES = ["te_zone_eui", "te_zone_elec", "te_zone_eui_area", "te_zone_eui_std", "te_zone_elec_area"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(N_THREADS)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
STARTED = time.monotonic()
ERRORS, TRAIN_EVENTS, DEPENDENCY_STATUS = [], [], []
FOLD_RESULTS = {name: {} for name in SPECS}


In [2]:
#tambah
def log(message):
    line = f"[{datetime.now(ZoneInfo('Asia/Jakarta')):%H:%M:%S}] {message}"
    print(line, flush=True)
    directory = globals().get("RUN_DIR")
    if directory is not None and Path(directory).is_dir():
        with (Path(directory) / "progress.log").open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")

def json_value(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Tidak bisa ditulis ke JSON: {type(value)}")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_value, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def write_npz(path, **values):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **values)
    temporary.replace(path)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def hash_array(value):
    return hashlib.sha256(memoryview(np.ascontiguousarray(value)).cast("B")).hexdigest()

def nrmse(actual, predicted):
    return np.sqrt(np.mean((np.asarray(predicted, np.float64) - actual) ** 2, axis=0)) / SIGMA

def atomic_torch_save(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def torch_load(path):
    return torch.load(path, map_location="cpu", weights_only=True)

def atomic_joblib(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(value, temporary, compress=0, protocol=5)
    temporary.replace(path)

@contextmanager
def heartbeat(label):
    stopped = threading.Event()
    started = time.monotonic()
    def report():
        while not stopped.wait(30):
            log(f"{label}: masih berjalan, {(time.monotonic() - started) / 60:.1f} menit.")
    worker = threading.Thread(target=report, daemon=True)
    worker.start()
    try:
        yield
    finally:
        stopped.set()
        worker.join(timeout=1)

INDEX, WHEEL_DIRS, visited = {}, set(), set()
for directory, subdirectories, filenames in os.walk(INPUT, followlinks=True):
    resolved = Path(directory).resolve()
    if resolved in visited:
        subdirectories[:] = []
        continue
    visited.add(resolved)
    for filename in filenames:
        if filename.endswith(".whl"):
            WHEEL_DIRS.add(Path(directory))
        if filename in ("train.csv", "prediksi_oof_test.npz", "reproduction_v21.json"):
            INDEX.setdefault(filename, []).append(Path(directory) / filename)

def find_inputs(filename):
    return sorted(INDEX.get(filename, []))

log(f"Device: {DEVICE}; GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'tidak tersedia'}")
if RUN_TRAINING and REQUIRE_GPU and DEVICE == "cpu":
    raise RuntimeError("V21 membutuhkan GPU untuk training default. Kaggle Settings > Accelerator > GPU, lalu Run All. Tidak ada fallback training CPU diam-diam.")

PINS = {"pytabkit": "1.7.3", "tabm": "0.0.3", "rtdl_num_embeddings": "0.0.12"}
missing = []
for package, version in PINS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")

if missing and INSTALL_MISSING:
    constraints = WORK / "v21_existing_packages.txt"
    frozen = []
    for package in ("torch", "numpy", "pandas", "scikit-learn", "scipy"):
        frozen.append(f"{package}=={importlib.metadata.version(package)}")
    constraints.write_text("\n".join(frozen) + "\n", encoding="utf-8")
    base_command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--retries", "0",
                    "--timeout", "10", "--constraint", str(constraints)]
    attempts = []
    if WHEEL_DIRS:
        local_command = base_command + ["--no-index"]
        for directory in sorted(WHEEL_DIRS):
            local_command += ["--find-links", str(directory)]
        attempts.append(local_command + missing)
    if ALLOW_PYPI:
        attempts.append(base_command + missing)
    for command in attempts:
        log("Memasang dependensi V21 dari " + ("wheel Input" if "--no-index" in command else "PyPI"))
        try:
            result = subprocess.run(command, capture_output=True, text=True, timeout=180)
            print((result.stdout + result.stderr)[-5000:], flush=True)
            if result.returncode == 0:
                break
        except subprocess.TimeoutExpired:
            log("Instalasi melewati batas 180 detik; proses instalasi dihentikan.")
    importlib.invalidate_caches()

RealMLP_TD_Regressor, tabm, rtdl = None, None, None
for package, module_name in (("pytabkit", "pytabkit"), ("tabm", "tabm"), ("rtdl_num_embeddings", "rtdl_num_embeddings")):
    try:
        version = importlib.metadata.version(package)
        if version != PINS[package]:
            raise ImportError(f"Versi tersedia {version}, diperlukan {PINS[package]}.")
        module = importlib.import_module(module_name)
        if package == "pytabkit":
            RealMLP_TD_Regressor = module.RealMLP_TD_Regressor
        elif package == "tabm":
            tabm = module
        else:
            rtdl = module
        DEPENDENCY_STATUS.append({"package": package, "version": version, "available": True})
    except Exception:
        error = traceback.format_exc()
        DEPENDENCY_STATUS.append({"package": package, "available": False, "error": error})
        log(f"{package} tidak tersedia: {error.splitlines()[-1]}")
ENABLED = {name: (RealMLP_TD_Regressor is not None if spec["kind"] == "realmlp" else tabm is not None and rtdl is not None)
           for name, spec in SPECS.items()}
VERSIONS = {name: importlib.metadata.version(name) for name in ("numpy", "pandas", "torch", "scikit-learn", "joblib")}
VERSIONS.update({row["package"]: row.get("version") for row in DEPENDENCY_STATUS})
display(pd.DataFrame(DEPENDENCY_STATUS))


[19:14:21] Device: cuda:0; GPU: Tesla T4
[19:14:21] Memasang dependensi V21 dari PyPI
vtx]==12.8.1; platform_system == "Linux"->torch>=2.0->pytabkit==1.7.3) (1.13.1.3)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.1 MB/s eta 0:00:00



,package,version,available
0,pytabkit,1.7.3,True
1,tabm,0.0.3,True
2,rtdl_num_embeddings,0.0.12,True


## Data, fitur, dan acuan

CSV dicocokkan dengan ID NPZ Dani. Label, luas, dan pemeriksaan identitas memakai float64; matriks model float32. Fitur statis S7_density dipertahankan lengkap (329 kolom), ditambah lima TE zona yang dibuat cross-fit per subzone. Native neural input juga menerima kategori zona. Tidak memakai ID bangunan sebagai fitur.

Imputasi dan preprocessing neural di-fit pada bagian training. Untuk RealMLP, imputasi serta encoding eksternal dibekukan dari inner-training saat library melakukan refit outer-train. Untuk TabM, preprocessing dibangun ulang pada outer-train setelah jumlah epoch terkunci.

In [3]:
#tambah
sources = [Path(S28_PRED_PATH)] if S28_PRED_PATH else find_inputs("prediksi_oof_test.npz")
unique_sources = {}
for path in sources:
    if path.is_file() and path.with_name("prediksi_meta.json").is_file():
        key = (file_hash(path), file_hash(path.with_name("prediksi_meta.json")))
        unique_sources.setdefault(key, path)
if len(unique_sources) != 1:
    raise FileNotFoundError(
        f"Ditemukan {len(unique_sources)} sumber S28 berbeda. Pasang NPZ dan JSON dari paket teman "
        f"dalam folder yang sama, atau isi S28_PRED_PATH. Kandidat: {[str(p) for p in sources]}"
    )
SOURCE_SHA, META_SHA = next(iter(unique_sources))
SOURCE_PATH = next(iter(unique_sources.values()))
META_PATH = SOURCE_PATH.with_name("prediksi_meta.json")
SOURCE_META = read_json(META_PATH)
required_keys = {"oof", "oof_base", "test", "test_nolookup", "id_train", "id_test", "fold", "targets", "sigmas"}
with np.load(SOURCE_PATH, allow_pickle=False) as archive:
    assert required_keys.issubset(archive.files), archive.files
    assert archive["targets"].astype(str).tolist() == TARGETS
    assert np.allclose(archive["sigmas"], SIGMA, rtol=0, atol=1e-10)
    S28_OOF = archive["oof"].astype(np.float64)
    S28_TEST = archive["test"].astype(np.float64)
    S28_TEST_NOLOOKUP = archive["test_nolookup"].astype(np.float64)
    TRAIN_IDS, TEST_IDS = archive["id_train"].astype(str), archive["id_test"].astype(str)
    FOLD_ID = archive["fold"].copy()
assert S28_OOF.shape == (429303, 5) and S28_TEST.shape == S28_TEST_NOLOOKUP.shape == (183987, 5)
assert TRAIN_IDS.shape == (429303,) and TEST_IDS.shape == (183987,) and FOLD_ID.shape == (429303,)
assert np.array_equal(np.unique(FOLD_ID), np.arange(5))
assert SOURCE_META["urutan_kolom"] == TARGETS
for values in (S28_OOF, S28_TEST, S28_TEST_NOLOOKUP):
    assert np.isfinite(values).all(), "Ada NaN/Inf pada prediksi sumber"
assert np.array_equal(S28_TEST[:, :4], S28_TEST_NOLOOKUP[:, :4])
log(f"Sumber S28: {SOURCE_PATH}")
display(pd.DataFrame([{"array": name, "shape": values.shape, "NaN": int(np.isnan(values).sum()),
    "Inf": int(np.isinf(values).sum())} for name, values in
    (("oof", S28_OOF), ("test", S28_TEST), ("test_nolookup", S28_TEST_NOLOOKUP))]))


[19:14:44] Sumber S28: /kaggle/input/datasets/madeyzzz/artefak-dani/paket_model_untuk_teman/prediksi_oof_test.npz


,array,shape,NaN,Inf
0,oof,"(429303, 5)",0,0
1,test,"(183987, 5)",0,0
2,test_nolookup,"(183987, 5)",0,0


In [4]:
#tambah
data_candidates = [Path(DATA_DIR_OVERRIDE)] if DATA_DIR_OVERRIDE else sorted({
    p.parent for p in find_inputs("train.csv")
    if (p.parent / "test.csv").is_file() and (p.parent / "sample_submission.csv").is_file()})
DATA_DIR = None
for directory in data_candidates:
    log(f"Mencocokkan ID CSV: {directory}")
    train_header = pd.read_csv(directory / "train.csv", nrows=0).columns
    if not set(["id", "subzone_id", "bld_dim_06", *TARGETS]).issubset(train_header):
        continue
    train_ids = pd.read_csv(directory / "train.csv", usecols=["id"], dtype={"id": "string"})["id"].to_numpy(dtype=str)
    test_ids = pd.read_csv(directory / "test.csv", usecols=["id"], dtype={"id": "string"})["id"].to_numpy(dtype=str)
    if np.array_equal(train_ids, TRAIN_IDS) and np.array_equal(test_ids, TEST_IDS):
        DATA_DIR = directory
        break
if DATA_DIR is None:
    raise FileNotFoundError(
        "CSV kompetisi dengan ID/urutan yang cocok belum ditemukan. Tambahkan data kompetisi melalui Add Input "
        f"atau isi DATA_DIR_OVERRIDE. Folder terdeteksi: {[str(p) for p in data_candidates]}"
    )


IDS = ["id", "subzone_id", "zone_region_id"]
def read_csv(path):
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    text = set(IDS + ["bld_dim_18", "socio_demog_016"])
    return pd.read_csv(path, dtype={c: "string" if c in text else "float64" for c in columns}, low_memory=False)

log("Membaca seluruh kolom CSV")
train, test = read_csv(DATA_DIR / "train.csv"), read_csv(DATA_DIR / "test.csv")
assert train.shape == (429303, 209) and test.shape == (183987, 204)
SAMPLE = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": "string"})
assert SAMPLE.columns.tolist() == ["id", *TARGETS]
assert np.array_equal(train["id"].to_numpy(dtype=str), TRAIN_IDS)
assert np.array_equal(test["id"].to_numpy(dtype=str), TEST_IDS)
assert np.array_equal(SAMPLE["id"].to_numpy(dtype=str), TEST_IDS)
assert train["id"].is_unique and test["id"].is_unique
assert not train[IDS].isna().any().any() and not test[IDS].isna().any().any()
Y = train[TARGETS].to_numpy(np.float64)
AREA, TEST_AREA = train["bld_dim_06"].to_numpy(np.float64), test["bld_dim_06"].to_numpy(np.float64)
assert np.isfinite(Y).all()
UNITS, TEST_UNITS = train["bld_dim_12"].to_numpy(np.float64), test["bld_dim_12"].to_numpy(np.float64)
for values in (AREA, TEST_AREA, UNITS, TEST_UNITS):
    assert np.isfinite(values).all() and (values > 0).all()
assert np.allclose(Y[:, 0], Y[:, 2] * AREA, rtol=1e-10, atol=1e-6)
SUBZONES = train["subzone_id"].to_numpy(dtype=str)
TEST_SUBZONES = test["subzone_id"].to_numpy(dtype=str)
ZONES, TEST_ZONES = train["zone_region_id"].to_numpy(dtype=str), test["zone_region_id"].to_numpy(dtype=str)
group_folds = pd.DataFrame({"subzone": SUBZONES, "fold": FOLD_ID}).groupby("subzone")["fold"].nunique()
assert group_folds.max() == 1, "Subzone melintasi fold sumber"
assert train.groupby("subzone_id")["zone_region_id"].nunique().max() == 1
FOLDS = [(np.flatnonzero(FOLD_ID != f), np.flatnonzero(FOLD_ID == f)) for f in range(5)]
adoption = train.groupby("subzone_id")[TARGETS[4]].agg(["min", "max", "mean"])
assert ((adoption["max"] - adoption["min"]) <= 1e-5).all()
lookup = test["subzone_id"].map(adoption["mean"]).to_numpy(float)
LOOKUP_MASK = np.isfinite(lookup)
assert np.allclose(S28_TEST[LOOKUP_MASK, 4], lookup[LOOKUP_MASK], rtol=0, atol=1e-5)
assert np.allclose(S28_TEST[~LOOKUP_MASK, 4], S28_TEST_NOLOOKUP[~LOOKUP_MASK, 4], rtol=0, atol=1e-5)
CATEGORY_MAPS = {
    "bld_dim_18": {"High (<0%)": 0, "Medium (0-6%)": 1, "Low (6-18%)": 2, "Very Low (>18%)": 3},
    "socio_demog_016": {"Rural": 0, "Urban": 1}}

assert (Y[:, 2] >= 0).all() and (Y[:, 2] > 0).any()
for frame in (train, test):
    for name, mapping in CATEGORY_MAPS.items():
        assert frame[name].notna().all() and not set(frame[name].unique()) - set(mapping), name
        frame[name] = frame[name].map(mapping).astype("float64")


[19:14:44] Mencocokkan ID CSV: /kaggle/input/competitions/dicefest-2026-machine-learning
[19:15:02] Membaca seluruh kolom CSV


In [5]:
#tambah
def ratio(a, b):
    a, b = (np.asarray(a, np.float32), np.asarray(b, np.float32))
    result = np.full(len(a), np.nan, dtype=np.float32)
    np.divide(a, b, out=result, where=np.isfinite(b) & (np.abs(b) > 1e-06))
    result[~np.isfinite(result)] = np.nan
    return result

def build_features(train_raw, test_raw):
    raw_cols = [c for c in train_raw if c not in IDS + TARGETS]
    known_constants = ['socio_demog_001', 'socio_demog_004', 'socio_demog_026', 'socio_demog_034', 'socio_demog_039', 'socio_demog_045', 'socio_demog_054', 'socio_demog_060', 'socio_demog_069', 'bld_era_01', 'env_climate_06']
    assert set(known_constants).issubset(raw_cols)
    assert (train_raw[known_constants].nunique(dropna=False) <= 1).all()
    assert train_raw['env_climate_02'].equals(train_raw['env_climate_04'])
    drop_cols = known_constants + ['env_climate_04']
    RAW_FEATURES = [c for c in raw_cols if c not in drop_cols]
    train_base = train_raw[RAW_FEATURES].astype('float32').copy()
    test_base = test_raw[RAW_FEATURES].astype('float32').copy()
    raw_add_train, raw_add_test = ({}, {})
    for col in RAW_FEATURES:
        if train_base[col].isna().any() or test_base[col].isna().any():
            raw_add_train[f'missing_{col}'] = train_base[col].isna().to_numpy(np.float32)
            raw_add_test[f'missing_{col}'] = test_base[col].isna().to_numpy(np.float32)
    hvac_cols = [c for c in RAW_FEATURES if c.startswith('hvac_dist_')]
    for frame, extra in ((train_base, raw_add_train), (test_base, raw_add_test)):
        extra['hvac_rest'] = (100.0 - frame[hvac_cols].sum(axis=1)).to_numpy(np.float32)
    s6_train, s6_test, s7_train, s7_test, density_train, density_test = ({}, {}, {}, {}, {}, {})
    for col in [c for c in RAW_FEATURES if c.startswith('bld_dim_') and c != 'bld_dim_18']:
        name = f'rt_{col}'
        s6_train[name], s6_test[name] = (ratio(train_base[col], AREA), ratio(test_base[col], TEST_AREA))
    for label, train_group, test_group in (('subzone', SUBZONES, TEST_SUBZONES), ('zone', ZONES, TEST_ZONES)):
        counts = pd.Series(np.concatenate([train_group, test_group])).value_counts()
        s6_train[f'{label}_size'] = pd.Series(train_group).map(counts).to_numpy(np.float32)
        s6_test[f'{label}_size'] = pd.Series(test_group).map(counts).to_numpy(np.float32)
    assert set(POPULAR12).issubset(train_base.columns)
    for left, right in combinations(POPULAR12, 2):
        name = f'pw_{left}_{right}'
        for frame, extra in ((train_base, s7_train), (test_base, s7_test)):
            a, b = (frame[left].to_numpy(), frame[right].to_numpy())
            extra[name] = (np.sign(a) * np.log1p(np.abs(a)) - np.sign(b) * np.log1p(np.abs(b))).astype(np.float32)
    all_zones = np.concatenate([ZONES, TEST_ZONES])
    for col in POPULAR12:
        values = np.concatenate([train_base[col].to_numpy(), test_base[col].to_numpy()])
        means = pd.DataFrame({'zone': all_zones, 'value': values}).groupby('zone')['value'].mean()
        for frame, zones, extra in ((train_base, ZONES, s7_train), (test_base, TEST_ZONES, s7_test)):
            group_mean = pd.Series(zones).map(means).to_numpy(np.float32)
            extra[f'za_mean_{col}'] = group_mean
            extra[f'za_diff_{col}'] = frame[col].to_numpy() - group_mean
    for col in ['bld_dim_06', 'bld_dim_07', 'bld_dim_08']:
        name = f'density_{col}_per_unit'
        density_train[name] = ratio(train_base[col], UNITS)
        density_test[name] = ratio(test_base[col], TEST_UNITS)
    STATIC_TRAIN = pd.concat([train_base, pd.DataFrame(raw_add_train), pd.DataFrame(s6_train), pd.DataFrame(s7_train), pd.DataFrame(density_train)], axis=1).astype('float32')
    STATIC_TEST = pd.concat([test_base, pd.DataFrame(raw_add_test), pd.DataFrame(s6_test), pd.DataFrame(s7_test), pd.DataFrame(density_test)], axis=1).astype('float32')
    STATIC_TRAIN.replace([np.inf, -np.inf], np.nan, inplace=True)
    STATIC_TEST.replace([np.inf, -np.inf], np.nan, inplace=True)
    FEATURE_SETS = {'raw': RAW_FEATURES + list(raw_add_train), 'S6': RAW_FEATURES + list(raw_add_train) + list(s6_train), 'S7': RAW_FEATURES + list(raw_add_train) + list(s6_train) + list(s7_train), 'S7_density': RAW_FEATURES + list(raw_add_train) + list(s6_train) + list(s7_train) + list(density_train)}
    return STATIC_TRAIN, STATIC_TEST, FEATURE_SETS

log("Membangun fitur statis S7_density dari seluruh fitur mentah")
STATIC_TRAIN, STATIC_TEST, FEATURE_SETS = build_features(train, test)
BASE_FEATURES = STATIC_TRAIN.columns.tolist()
ALL_FEATURES = BASE_FEATURES + TE_NAMES
CAT_FEATURES = ["bld_dim_18", "socio_demog_016"]
assert BASE_FEATURES == STATIC_TEST.columns.tolist()
assert len(BASE_FEATURES) == 329 and len(ALL_FEATURES) == 334
assert {"bld_dim_20", "bld_dim_32", "bld_dim_43", "socio_demog_005", "socio_demog_038", "socio_demog_082"}.issubset(BASE_FEATURES)
STATIC_STATE = {"popular12": POPULAR12, "category_maps": CATEGORY_MAPS,
    "zone_sizes": {str(k): int(v) for k, v in pd.Series(np.concatenate([ZONES, TEST_ZONES])).value_counts().items()},
    "subzone_sizes": {str(k): int(v) for k, v in pd.Series(np.concatenate([SUBZONES, TEST_SUBZONES])).value_counts().items()},
    "zone_means": {}, "feature_names": BASE_FEATURES}
for name in POPULAR12:
    values = np.concatenate([train[name].to_numpy(), test[name].to_numpy()])
    means = pd.DataFrame({"zone": np.concatenate([ZONES, TEST_ZONES]), "value": values}).groupby("zone")["value"].mean().dropna()
    STATIC_STATE["zone_means"][name] = {str(k): float(v) for k, v in means.items()}
X, TEST_X = STATIC_TRAIN.to_numpy(copy=True), STATIC_TEST.to_numpy(copy=True)
assert not np.isinf(X).any() and not np.isinf(TEST_X).any()
missing = pd.DataFrame({"train_NaN": STATIC_TRAIN.isna().sum(), "test_NaN": STATIC_TEST.isna().sum()})
display(missing.loc[(missing > 0).any(axis=1)])
del STATIC_TRAIN, STATIC_TEST, train, test
gc.collect()
log(f"Fitur statis {X.shape}; fitur model {len(ALL_FEATURES)}; imputasi dilakukan per job neural.")


[19:15:40] Membangun fitur statis S7_density dari seluruh fitur mentah


,train_NaN,test_NaN
bld_dim_04,695,336
bld_dim_07,695,336
rt_bld_dim_04,695,336
rt_bld_dim_07,695,336
pw_bld_dim_01_bld_dim_04,695,336
pw_bld_dim_01_bld_dim_07,695,336
pw_bld_dim_02_bld_dim_04,695,336
pw_bld_dim_02_bld_dim_07,695,336
pw_bld_dim_04_bld_dim_06,695,336
pw_bld_dim_04_bld_dim_07,695,336


[19:15:49] Fitur statis (429303, 329); fitur model 334; imputasi dilakukan per job neural.


In [6]:
#tambah
def cross_parameters(indices):
    gas, eui = S28_OOF[indices, 0], S28_OOF[indices, 2]
    differences = (AREA[indices] * eui - gas, gas / AREA[indices] - eui)
    values = []
    for target, difference in zip((0, 2), differences):
        denominator = float(np.dot(difference, difference))
        alpha = -float(np.dot(S28_OOF[indices, target] - Y[indices, target], difference)) / denominator if denominator else 0.0
        values.append(float(np.clip(alpha, 0.0, 1.0)))
    return values

def cross_apply(predictions, area, weights):
    result = predictions.copy()
    result[:, 0] = predictions[:, 0] + weights[0] * (area * predictions[:, 2] - predictions[:, 0])
    result[:, 2] = predictions[:, 2] + weights[1] * (predictions[:, 0] / area - predictions[:, 2])
    result[:, [0, 2]] = np.maximum(result[:, [0, 2]], 0.0)
    return result

REF_OOF = S28_OOF.copy()
REFERENCE_WEIGHTS = []
for fold, (fit, valid) in enumerate(FOLDS):
    weights = cross_parameters(fit)
    REF_OOF[valid] = cross_apply(S28_OOF[valid], AREA[valid], weights)
    REFERENCE_WEIGHTS.append({"fold": fold + 1, "weights": weights})
FULL_REFERENCE_WEIGHTS = cross_parameters(np.arange(len(Y)))
REF_TEST = cross_apply(S28_TEST, TEST_AREA, FULL_REFERENCE_WEIGHTS)
REF_NOLOOKUP = cross_apply(S28_TEST_NOLOOKUP, TEST_AREA, FULL_REFERENCE_WEIGHTS)
REF_SCORES = nrmse(Y, REF_OOF)
REF_CV = float(REF_SCORES.mean())
assert np.isclose(nrmse(Y, S28_OOF).mean(), SOURCE_META["cv_gabungan_tanpa_lookup"], rtol=0, atol=1e-5)
log(f"Acuan v17: CV {REF_CV:.9f}; public historis yang dilaporkan 0,25176")
if not np.isclose(REF_CV, 0.26950079729861065, rtol=0, atol=1e-7):
    log("CV berbeda dari run 0,25176; skor historis belum terpetakan untuk sumber ini.")
display(pd.DataFrame({"target": TARGETS, "RMSE": REF_SCORES * SIGMA, "NRMSE": REF_SCORES}))

REFERENCE_PUBLIC_MATCH = bool(np.isclose(REF_CV, 0.26950079729861065, rtol=0, atol=1e-7))
REFERENCE_PUBLIC = 0.25176 if REFERENCE_PUBLIC_MATCH else None


[19:15:49] Acuan v17: CV 0.269500797; public historis yang dilaporkan 0,25176


,target,RMSE,NRMSE
0,target_macro_gas_kwh,37972.587579,2.502312e-01
1,target_macro_elec_kwh,13325.729144,3.193675e-01
2,target_eui_gas_kwh_per_sqm,26.832733,6.600918e-01
3,target_clean_building_ratio,0.000001,1.066132e-07
4,target_electric_heating_adoption,0.845900,1.178133e-01


In [7]:
#tambah
DATA_HASHES = {name: file_hash(DATA_DIR / name) for name in ("train.csv", "test.csv", "sample_submission.csv")}
REQUEST = {"config": CONFIG, "specs": SPECS, "source_sha256": SOURCE_SHA, "metadata_sha256": META_SHA,
           "data_hashes": DATA_HASHES, "feature_names": ALL_FEATURES, "versions": VERSIONS,
           "device_name": torch.cuda.get_device_name(0) if DEVICE != "cpu" else "cpu"}
RUN_SIGNATURE = hashlib.sha256(json.dumps(REQUEST, sort_keys=True).encode()).hexdigest()
RUN_DIR = WORK / "outputs" / "dicefest_v21" / RUN_SIGNATURE[:16]
for name in ("models", "reports", "predictions", "parameters", "reference", "inputs", "cache", "submissions"):
    (RUN_DIR / name).mkdir(parents=True, exist_ok=True)
reuse_paths = [Path(REUSE_RUN_DIR) / "reproduction_v21.json"] if REUSE_RUN_DIR else find_inputs("reproduction_v21.json")
for path in reuse_paths:
    if path.is_file() and read_json(path).get("signature") == RUN_SIGNATURE:
        source_directory = path.parent
        for name in ("models", "cache"):
            if (source_directory / name).is_dir() and (source_directory / name).resolve() != (RUN_DIR / name).resolve():
                shutil.copytree(source_directory / name, RUN_DIR / name, dirs_exist_ok=True)
        log(f"Cache dengan signature sama ditemukan: {source_directory}")
        break
write_json(RUN_DIR / "feature_manifest.json", {"static": STATIC_STATE, "all_features": ALL_FEATURES,
    "categorical_features": CAT_FEATURES, "extra_categorical_feature": "zone_category",
    "zone_encoding": "cross-fit inner subzone; outer labels excluded", "encoding_folds": CONFIG["encoding_folds"]})
write_json(RUN_DIR / "dependencies.json", DEPENDENCY_STATUS)
write_json(RUN_DIR / "source_manifest.json", {"signature": RUN_SIGNATURE, **REQUEST})
write_json(RUN_DIR / "reproduction_v21.json", {"signature": RUN_SIGNATURE, "config": CONFIG,
    "source_sha256": SOURCE_SHA, "data_hashes": DATA_HASHES, "target_order": TARGETS,
    "fold_group": "subzone_id", "fold_hash": hash_array(FOLD_ID), "reference_cv": REF_CV,
    "versions": VERSIONS, "status": "started", "evaluation": CONFIG["evaluation"]})
write_npz(RUN_DIR / "reference" / "reference_v17.npz", oof=REF_OOF, test=REF_NOLOOKUP,
          test_lookup=REF_TEST, train_ids=TRAIN_IDS, test_ids=TEST_IDS, fold_id=FOLD_ID, target_names=np.array(TARGETS))
write_json(RUN_DIR / "reference" / "reference_v17.json", {"fold_weights": REFERENCE_WEIGHTS,
    "test_weights": FULL_REFERENCE_WEIGHTS, "CV": REF_CV, "historical_public_reported": REFERENCE_PUBLIC,
    "historical_cv_match": REFERENCE_PUBLIC_MATCH, "current_public": None})
write_npz(RUN_DIR / "inputs" / "data_context.npz", y=Y, area=AREA, test_area=TEST_AREA,
          units=UNITS, test_units=TEST_UNITS, zones=ZONES, test_zones=TEST_ZONES,
          subzones=SUBZONES, test_subzones=TEST_SUBZONES, train_ids=TRAIN_IDS, test_ids=TEST_IDS,
          fold_id=FOLD_ID, target_names=np.array(TARGETS), sigma=SIGMA)
write_json(RUN_DIR / "parameters" / "specs.json", SPECS)
log(f"Signature {RUN_SIGNATURE[:16]}; output {RUN_DIR}")


[19:16:00] Signature 926c7d3537eabbb2; output /kaggle/working/outputs/dicefest_v21/926c7d3537eabbb2


In [8]:
#tambah
def zone_state(indices):
    eui, elec, area = Y[indices, 2], Y[indices, 1], AREA[indices]
    table = pd.DataFrame({"zone": ZONES[indices], "eui": eui, "elec": elec,
        "eui2": eui * eui, "area": area, "eui_area": eui * area, "elec_area": elec / area})
    grouped = table.groupby("zone").agg(n=("eui", "size"), eui=("eui", "sum"), elec=("elec", "sum"),
        eui2=("eui2", "sum"), area=("area", "sum"), eui_area=("eui_area", "sum"), elec_area=("elec_area", "sum"))
    alpha = CONFIG["zone_smoothing"]
    priors = np.array([eui.mean(), elec.mean(), np.sum(eui * area) / area.sum(),
                       eui.std(ddof=0), np.mean(elec / area)], dtype=np.float64)
    means = (grouped["eui"] + alpha * priors[0]) / (grouped["n"] + alpha)
    moment = (grouped["eui2"] + alpha * np.mean(eui * eui)) / (grouped["n"] + alpha)
    values = np.column_stack((means,
        (grouped["elec"] + alpha * priors[1]) / (grouped["n"] + alpha),
        (grouped["eui_area"] + alpha * area.mean() * priors[2]) / (grouped["area"] + alpha * area.mean()),
        np.sqrt(np.maximum(moment - means ** 2, 0.0)),
        (grouped["elec_area"] + alpha * priors[4]) / (grouped["n"] + alpha)))
    return {"priors": priors.tolist(), "zones": {str(zone): row.tolist() for zone, row in zip(grouped.index, values)},
        "feature_names": ALL_FEATURES, "base_features": BASE_FEATURES, "te_names": TE_NAMES,
        "categorical_features": CAT_FEATURES, "category_maps": CATEGORY_MAPS,
        "target": TARGETS[2], "numeric_missing": "fit-only median; neural preprocessing stored per fold"}

def query_encoding(zones, state):
    values = np.asarray([state["zones"].get(str(zone), state["priors"]) for zone in zones], dtype=np.float32)
    assert values.shape == (len(zones), len(TE_NAMES)) and np.isfinite(values).all()
    return values

def prepare_fit(fit):
    groups, inverse = np.unique(SUBZONES[fit], return_inverse=True)
    assert len(groups) >= CONFIG["encoding_folds"]
    order = np.random.default_rng(SEED).permutation(len(groups))
    assignments = np.empty(len(groups), dtype=np.int8)
    assignments[order] = np.arange(len(groups)) % CONFIG["encoding_folds"]
    inner_id = assignments[inverse]
    encoded = np.full((len(fit), len(TE_NAMES)), np.nan, dtype=np.float32)
    for fold in range(CONFIG["encoding_folds"]):
        mask = inner_id == fold
        state = zone_state(fit[~mask])
        encoded[mask] = query_encoding(ZONES[fit[mask]], state)
    assert np.isfinite(encoded).all()
    state = zone_state(fit)
    values = np.column_stack((X[fit], encoded))
    assert not np.isinf(values).any()
    return np.ascontiguousarray(values, dtype=np.float32), state

def prepare_query(base_values, zones, state):
    values = np.column_stack((base_values, query_encoding(zones, state)))
    assert values.shape[1] == len(state["feature_names"]) and not np.isinf(values).any()
    return np.ascontiguousarray(values, dtype=np.float32)


In [9]:
#tambah
def inner_split(fold):
    outer, valid = FOLDS[fold]
    groups = np.unique(SUBZONES[outer])
    order = np.random.default_rng(SEED + fold).permutation(len(groups))
    stop_groups = groups[order[:max(1, int(np.ceil(CONFIG["inner_group_fraction"] * len(groups))))]]
    stop_mask = np.isin(SUBZONES[outer], stop_groups)
    learn, stop = outer[~stop_mask], outer[stop_mask]
    assert len(learn) and len(stop)
    assert set(SUBZONES[learn]).isdisjoint(SUBZONES[stop])
    assert set(SUBZONES[outer]).isdisjoint(SUBZONES[valid])
    return outer, valid, learn, stop, np.flatnonzero(stop_mask)

def real_frame(values, zones, prep):
    values = np.asarray(values, np.float32)
    values = np.where(np.isfinite(values), values, prep["median"]).astype(np.float32)
    frame = pd.DataFrame(values, columns=ALL_FEATURES)
    for name, cardinality in (("bld_dim_18", 4), ("socio_demog_016", 2)):
        frame[name] = pd.Categorical(frame[name].astype(np.int32), categories=range(cardinality))
    codes = pd.Index(prep["zone_levels"]).get_indexer(zones)
    codes[codes < 0] = len(prep["zone_levels"])
    frame["zone_category"] = pd.Categorical(codes, categories=range(len(prep["zone_levels"]) + 1))
    assert np.isfinite(frame.drop(columns=[*CAT_FEATURES, "zone_category"]).to_numpy()).all()
    return frame

def predict_real(model, base_values, zones, prep, label):
    result = np.empty(len(base_values), dtype=np.float64)
    started = time.monotonic()
    for begin in range(0, len(base_values), CONFIG["realmlp"]["query_chunk_rows"]):
        end = min(begin + CONFIG["realmlp"]["query_chunk_rows"], len(base_values))
        values = prepare_query(base_values[begin:end], zones[begin:end], prep["zone_state"])
        frame = real_frame(values, zones[begin:end], prep)
        result[begin:end] = np.asarray(model.predict(frame)).reshape(-1)
        if time.monotonic() - started >= 30:
            log(f"{label}: prediksi {end:,}/{len(base_values):,} baris.")
            started = time.monotonic()
    return result

def train_realmlp_fold(name, fold, directory):
    outer, valid, learn, stop, stop_positions = inner_split(fold)
    log(f"{name} fold {fold}: membangun preprocessing inner-training.")
    with heartbeat(f"{name} fold {fold} preprocessing"):
        learn_values, encoding = prepare_fit(learn)
        stop_values = prepare_query(X[stop], ZONES[stop], encoding)
        median = np.nanmedian(learn_values, axis=0).astype(np.float32)
        median[~np.isfinite(median)] = 0.0
        prep = {"median": median, "zone_state": encoding, "zone_levels": np.unique(ZONES[learn]),
                "all_features": ALL_FEATURES, "categorical_features": [*CAT_FEATURES, "zone_category"],
                "external_prep_fitted_on": "inner_learning_only_and_frozen_during_native_refit"}
        values = np.empty((len(outer), len(ALL_FEATURES)), dtype=np.float32)
        learn_mask = np.ones(len(outer), dtype=bool)
        learn_mask[stop_positions] = False
        values[learn_mask], values[stop_positions] = learn_values, stop_values
        frame = real_frame(values, ZONES[outer], prep)
    del learn_values, stop_values, values
    target = SPECS[name]["target"]
    label = Y[outer, target].copy()
    if SPECS[name]["transform"] == "divide_by_area":
        label /= AREA[outer]
    parameters = {"device": DEVICE, "random_state": SEED + fold, "n_cv": 1, "n_refit": 1,
        "n_repeats": 1, "n_threads": N_THREADS, "verbosity": CONFIG["realmlp"]["verbosity"],
        "n_epochs": CONFIG["realmlp"]["n_epochs"], "batch_size": CONFIG["realmlp"]["batch_size"],
        "predict_batch_size": CONFIG["realmlp"]["predict_batch_size"], "val_metric_name": "rmse", "train_metric_name": "mse",
        "use_early_stopping": True, "early_stopping_additive_patience": CONFIG["realmlp"]["patience"],
        "early_stopping_multiplicative_patience": 1.0, "tmp_folder": str(directory / "native_training")}
    model = RealMLP_TD_Regressor(**parameters)
    log(f"{name} fold {fold}: seleksi epoch inner-subzone, lalu native refit seluruh {len(outer):,} outer-train.")
    with heartbeat(f"{name} fold {fold} native training/refit"):
        model.fit(frame, label, val_idxs=stop_positions, cat_col_names=[*CAT_FEATURES, "zone_category"])
    del frame, label
    gc.collect()
    prep_path, model_path = directory / "preprocessor.joblib", directory / "realmlp_model.joblib"
    atomic_joblib(prep_path, prep)
    model.to("cpu")
    log(f"{name} fold {fold}: menyimpan estimator resmi.")
    atomic_joblib(model_path, model)
    model.to(DEVICE)
    with heartbeat(f"{name} fold {fold} prediksi"):
        pv = predict_real(model, X[valid], ZONES[valid], prep, f"{name} valid")
        pt = predict_real(model, TEST_X, TEST_ZONES, prep, f"{name} test")
    info = {"parameters": parameters, "fit_params": getattr(model, "fit_params_", {}),
            "inner_learn_rows": len(learn), "inner_stop_rows": len(stop), "outer_train_rows": len(outer),
            "native_refit": True, "early_stopping_target": "electricity_per_area_RMSE" if target == 1 else "raw_EUI_RMSE",
            "sample_weights": "none; official estimator does not expose sample_weight",
            "native_model_format": "joblib official pytabkit estimator; includes internal fitted preprocessing",
            "interrupted_training_resume": False, "completed_fold_reuse": True}
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"valid": pv, "test": pt, "model_path": model_path, "prep_path": prep_path, "training_info": info}


## Training TabM-PWL

Embedding numerik memakai batas quantile dari bagian training dan versi B resmi. Loss dirata-ratakan atas error setiap anggota; prediksi anggota baru dirata-ratakan saat evaluasi. Inner holdout memilih epoch dengan RMSE EUI mentah. Outer refit memakai epoch tersebut dan preprocessing baru dari outer-train.

Checkpoint TabM menyimpan optimizer dan RNG agar bisa melanjutkan dari epoch lengkap terakhir. RealMLP menyimpan estimator resmi setelah fit selesai; training RealMLP yang terputus perlu diulang untuk fold tersebut. Hasil fold lengkap kedua jenis model bisa dipakai ulang.

In [10]:
#tambah
def v21t_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: v21t_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [v21t_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(v21t_cpu(item) for item in value)
    return value


def v21t_categories(values, zones):
    columns = [pd.Series(values[:, ALL_FEATURES.index(name)]).astype(str)
               for name in CAT_FEATURES]
    columns.append(pd.Series(np.asarray(zones, dtype=str)))
    return columns


def v21t_transform(values, zones, prep):
    numerical = np.asarray(values[:, prep["numeric_positions"]], dtype=np.float32).copy()
    missing_rows, missing_columns = np.where(~np.isfinite(numerical))
    numerical[missing_rows, missing_columns] = prep["medians"][missing_columns]
    numerical -= prep["centers"]
    numerical /= prep["scales"]
    categorical = np.empty((len(values), len(prep["category_maps"])), dtype=np.int64)
    for column, (series, mapping) in enumerate(zip(v21t_categories(values, zones), prep["category_maps"])):
        categorical[:, column] = series.map(mapping).fillna(len(mapping)).to_numpy(np.int64)
    if not np.isfinite(numerical).all():
        raise ValueError("Transformasi fitur TabM menghasilkan NaN/Inf.")
    return np.ascontiguousarray(numerical), categorical


def v21t_prepare(fit_indices, directory):
    prep_path = directory / "prep.joblib"
    values, encoding_state = prepare_fit(fit_indices)
    values = np.asarray(values, dtype=np.float32)
    if values.shape[1] != len(ALL_FEATURES):
        raise ValueError("Urutan fitur TabM tidak cocok dengan ALL_FEATURES.")
    fit_hash = hash_array(np.asarray(fit_indices, dtype=np.int64))
    feature_hash = hash_array(values)
    if prep_path.is_file():
        prep = joblib.load(prep_path)
        if prep["fit_hash"] != fit_hash or prep["feature_hash"] != feature_hash:
            raise ValueError("Fitur training berubah; checkpoint fase TabM tidak boleh dipakai ulang.")
        if prep["feature_names"] != list(ALL_FEATURES):
            raise ValueError("Urutan kolom checkpoint TabM berbeda.")
    else:
        category_positions = {ALL_FEATURES.index(name) for name in CAT_FEATURES}
        positions = np.asarray([i for i in range(values.shape[1]) if i not in category_positions], dtype=np.int64)
        numerical = values[:, positions].copy()
        numerical[~np.isfinite(numerical)] = np.nan
        medians = np.nanmedian(numerical, axis=0)
        medians = np.where(np.isfinite(medians), medians, 0.0).astype(np.float32)
        missing_rows, missing_columns = np.where(~np.isfinite(numerical))
        numerical[missing_rows, missing_columns] = medians[missing_columns]
        deviations = numerical.std(axis=0, dtype=np.float64)
        keep = np.isfinite(deviations) & (deviations > 0)
        if not keep.any():
            raise ValueError("Tidak ada fitur numerik TabM yang bervariasi pada training.")
        numerical = numerical[:, keep]
        medians = medians[keep]
        centers = np.median(numerical, axis=0).astype(np.float32)
        lower, upper = np.quantile(numerical, [0.25, 0.75], axis=0)
        spread = upper - lower
        scales = np.where(spread >= 1e-3 * deviations[keep], spread, deviations[keep]).astype(np.float32)
        scales = np.maximum(scales, np.float32(1e-12))
        category_maps = [{value: i for i, value in enumerate(sorted(series.unique().tolist()))}
                         for series in v21t_categories(values, ZONES[fit_indices])]
        target = Y[fit_indices, 2].astype(np.float64)
        target_mean, target_scale = float(target.mean()), float(target.std())
        if target_scale <= 0 or not np.isfinite(target_scale):
            target_scale = 1.0
        prep = {"feature_names": list(ALL_FEATURES), "fit_hash": fit_hash,
                "feature_hash": feature_hash, "encoding_state": encoding_state,
                "numeric_positions": positions[keep], "medians": medians,
                "centers": centers, "scales": scales, "category_maps": category_maps,
                "cat_cardinalities": [len(mapping) + 1 for mapping in category_maps],
                "target_mean": target_mean, "target_scale": target_scale}
        scaled, _ = v21t_transform(values, ZONES[fit_indices], prep)
        log(f"TabM: menghitung batas PWL dari {len(fit_indices):,} baris training.")
        prep["bins"] = [item.cpu() for item in rtdl.compute_bins(
            torch.from_numpy(scaled), n_bins=int(CONFIG["tabm"]["n_bins"]))]
        prep_temp = prep_path.with_suffix(".joblib.partial")
        joblib.dump(prep, prep_temp, compress=0)
        prep_temp.replace(prep_path)
        del numerical, scaled
    numerical, categorical = v21t_transform(values, ZONES[fit_indices], prep)
    del values
    return numerical, categorical, prep


def v21t_model(prep):
    settings = CONFIG["tabm"]
    embeddings = rtdl.PiecewiseLinearEmbeddings(
        prep["bins"], int(settings["d_embedding"]), activation=False, version="B")
    return tabm.TabM.make(
        n_num_features=len(prep["numeric_positions"]),
        cat_cardinalities=prep["cat_cardinalities"], num_embeddings=embeddings,
        d_out=1, k=int(settings["k"]), n_blocks=int(settings["n_blocks"]),
        d_block=int(settings["d_block"]), dropout=float(settings["dropout"])).to(DEVICE)


def v21t_predict_scaled(model, numerical, categorical, target_mean, target_scale):
    model.eval()
    predictions = []
    batch_size = int(CONFIG["tabm"]["predict_batch_size"])
    use_amp = str(DEVICE).startswith("cuda")
    with torch.inference_mode():
        for start in range(0, len(numerical), batch_size):
            x_num = torch.from_numpy(numerical[start:start + batch_size]).to(DEVICE)
            x_cat = torch.from_numpy(categorical[start:start + batch_size]).to(DEVICE)
            with torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp):
                output = model(x_num, x_cat)
            predictions.append(output.float().mean(dim=1).squeeze(-1).cpu().numpy())
    result = np.concatenate(predictions).astype(np.float64) * target_scale + target_mean
    if not np.isfinite(result).all():
        raise ValueError("Prediksi TabM menghasilkan NaN/Inf.")
    return result


def v21t_predict_query(model, prep, base_values, zones):
    values = prepare_query(base_values, zones, prep["encoding_state"])
    numerical, categorical = v21t_transform(np.asarray(values, dtype=np.float32), zones, prep)
    return v21t_predict_scaled(model, numerical, categorical, prep["target_mean"], prep["target_scale"])


def v21t_phase(fold, phase, fit_indices, stop_indices, directory, fixed_epochs=None):
    directory.mkdir(parents=True, exist_ok=True)
    phase_started = time.monotonic()
    settings = CONFIG["tabm"]
    numerical, categorical, prep = v21t_prepare(fit_indices, directory)
    torch.manual_seed(SEED + 1009 * int(fold) + (0 if phase == "inner" else 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 1009 * int(fold) + (0 if phase == "inner" else 1))
    model = v21t_model(prep)
    model_path, last_path = directory / "model.pt", directory / "last.pt"
    if model_path.is_file():
        saved = torch_load(model_path)
        model.load_state_dict(saved["state_dict"])
        log(f"TabM fold {fold} {phase}: fase lengkap dipakai ulang.")
        return model, prep, saved["training_info"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(settings["lr"]),
                                  weight_decay=float(settings["weight_decay"]))
    use_amp = str(DEVICE).startswith("cuda")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    generator = torch.Generator(device="cpu").manual_seed(SEED + 1009 * int(fold) + 17)
    maximum_epochs = int(fixed_epochs if fixed_epochs is not None else settings["epochs"])
    next_epoch, best_epoch, best_rmse, stalled = 0, 0, float("inf"), 0
    history = []
    elapsed_previous = 0.0
    if last_path.is_file():
        last = torch_load(last_path)
        if last["fixed_epochs"] != fixed_epochs:
            raise ValueError("Jumlah epoch refit TabM berubah dari checkpoint.")
        model.load_state_dict(last["state_dict"])
        optimizer.load_state_dict(last["optimizer"])
        scaler.load_state_dict(last["scaler"])
        generator.set_state(last["shuffle_rng"])
        torch.set_rng_state(last["torch_rng"])
        if use_amp and last["cuda_rng"]:
            torch.cuda.set_rng_state_all(last["cuda_rng"])
        next_epoch, best_epoch = int(last["next_epoch"]), int(last["best_epoch"])
        best_rmse, stalled = float(last["best_rmse"]), int(last["stalled"])
        history, elapsed_previous = last["history"], float(last["seconds"])
        log(f"TabM fold {fold} {phase}: lanjut setelah epoch {next_epoch} yang sudah lengkap.")
    x_num_cpu, x_cat_cpu = torch.from_numpy(numerical), torch.from_numpy(categorical)
    target = ((Y[fit_indices, 2] - prep["target_mean"]) / prep["target_scale"]).astype(np.float32)
    y_cpu = torch.from_numpy(target)
    stop_num = stop_cat = None
    if stop_indices is not None:
        stop_values = prepare_query(X[stop_indices], ZONES[stop_indices], prep["encoding_state"])
        stop_num, stop_cat = v21t_transform(np.asarray(stop_values, dtype=np.float32), ZONES[stop_indices], prep)
        del stop_values
    batch_size = int(settings["batch_size"])
    stage_limit = settings.get("max_seconds_per_phase")
    stage_limit = float(stage_limit) if stage_limit is not None else None
    best_path = directory / "best.pt"
    already_stopped = phase == "inner" and stalled >= int(settings["patience"])
    limit_text = f", batas {stage_limit / 60:.0f} menit" if stage_limit is not None else ""
    log(f"TabM fold {fold} {phase}: {len(fit_indices):,} baris, {len(prep['numeric_positions'])} numerik, "
        f"maksimal {maximum_epochs} epoch{limit_text}.")
    for epoch in range(next_epoch, maximum_epochs):
        if already_stopped:
            break
        epoch_started = time.monotonic()
        model.train()
        order = torch.randperm(len(fit_indices), generator=generator)
        squared_loss, seen = 0.0, 0
        for start in range(0, len(order), batch_size):
            if stage_limit is not None and time.monotonic() - phase_started > stage_limit:
                raise TimeoutError(f"TabM fold {fold} {phase} mencapai batas waktu; "
                                   "checkpoint epoch lengkap tersedia untuk melanjutkan.")
            batch = order[start:start + batch_size]
            x_num, x_cat, y_batch = x_num_cpu[batch].to(DEVICE), x_cat_cpu[batch].to(DEVICE), y_cpu[batch].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp):
                predicted = model(x_num, x_cat).squeeze(-1)
                loss = ((predicted.float() - y_batch[:, None]) ** 2).mean()
            if not torch.isfinite(loss).item():
                raise ValueError("Loss TabM tidak finite; checkpoint epoch lengkap sebelumnya dipertahankan.")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            squared_loss += float(loss.detach().cpu()) * len(batch)
            seen += len(batch)
        row = {"epoch": epoch + 1, "train_mse_standardized": squared_loss / seen,
               "seconds": time.monotonic() - epoch_started}
        if stop_indices is not None:
            stop_prediction = v21t_predict_scaled(model, stop_num, stop_cat, prep["target_mean"], prep["target_scale"])
            rmse = float(np.sqrt(np.mean((stop_prediction - Y[stop_indices, 2]) ** 2)))
            row["inner_rmse_eui"] = rmse
            if rmse < best_rmse:
                best_epoch, best_rmse, stalled = epoch + 1, rmse, 0
                atomic_torch_save(best_path, {"state_dict": v21t_cpu(model.state_dict()),
                                             "best_epoch": best_epoch, "inner_rmse_eui": best_rmse})
            else:
                stalled += 1
        else:
            best_epoch = epoch + 1
        history.append(row)
        completed = epoch + 1 >= maximum_epochs or (phase == "inner" and stalled >= int(settings["patience"]))
        atomic_torch_save(last_path, {
            "state_dict": v21t_cpu(model.state_dict()), "optimizer": v21t_cpu(optimizer.state_dict()),
            "scaler": scaler.state_dict(), "shuffle_rng": generator.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": [state.cpu() for state in torch.cuda.get_rng_state_all()] if use_amp else [],
            "next_epoch": epoch + 1, "best_epoch": best_epoch, "best_rmse": best_rmse,
            "stalled": stalled, "fixed_epochs": fixed_epochs, "history": history,
            "seconds": elapsed_previous + time.monotonic() - phase_started, "complete": completed})
        pd.DataFrame(history).to_csv(directory / "epochs.csv", index=False)
        suffix = f", inner RMSE {row['inner_rmse_eui']:.6f}, terbaik epoch {best_epoch}" if stop_indices is not None else ""
        log(f"TabM fold {fold} {phase}: epoch {epoch + 1}/{maximum_epochs}, "
            f"MSE {row['train_mse_standardized']:.6f}{suffix}, {row['seconds']:.1f} detik.")
        if completed:
            break
    if best_epoch < 1:
        raise RuntimeError("Belum ada satu epoch TabM yang selesai; jalankan ulang untuk melanjutkan.")
    if phase == "inner":
        best = torch_load(best_path)
        model.load_state_dict(best["state_dict"])
        best_epoch, best_rmse = int(best["best_epoch"]), float(best["inner_rmse_eui"])
    info = {"phase": phase, "best_epoch": best_epoch,
            "inner_rmse_eui": best_rmse if stop_indices is not None else None,
            "completed_epochs": len(history), "fixed_epochs": fixed_epochs,
            "seconds": elapsed_previous + time.monotonic() - phase_started,
            "numeric_features": len(prep["numeric_positions"]), "cat_cardinalities": prep["cat_cardinalities"],
            "amp": use_amp, "resume": "Model, optimizer, scaler dan RNG pada batas epoch lengkap."}
    architecture = {key: settings[key] for key in ("k", "n_blocks", "d_block", "dropout", "d_embedding")}
    atomic_torch_save(model_path, {"state_dict": v21t_cpu(model.state_dict()), "architecture": architecture,
                                  "bins": prep["bins"], "n_num_features": len(prep["numeric_positions"]),
                                  "cat_cardinalities": prep["cat_cardinalities"], "training_info": info})
    write_json(directory / "training_info.json", info)
    del optimizer, scaler, numerical, categorical, x_num_cpu, x_cat_cpu, y_cpu
    gc.collect()
    return model, prep, info


def train_tabm_fold(name, fold, directory):
    directory = Path(directory)
    outer_fit, valid, learn, stop, _ = inner_split(fold)
    with heartbeat(f"{name} fold {fold}: seleksi epoch TabM-PWL"):
        inner_model, inner_prep, inner_info = v21t_phase(
            fold, "inner", learn, stop, directory / "inner")
    best_epoch = int(inner_info["best_epoch"])
    del inner_model, inner_prep
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    with heartbeat(f"{name} fold {fold}: refit {best_epoch} epoch TabM-PWL"):
        model, prep, refit_info = v21t_phase(
            fold, "refit", outer_fit, None, directory / "refit", fixed_epochs=best_epoch)
    with heartbeat(f"{name} fold {fold}: prediksi validasi dan test"):
        valid_prediction = v21t_predict_query(model, prep, X[valid], ZONES[valid])
        test_prediction = v21t_predict_query(model, prep, TEST_X, TEST_ZONES)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"valid": np.maximum(valid_prediction, 0.0), "test": np.maximum(test_prediction, 0.0),
            "model_path": str(directory / "refit" / "model.pt"),
            "prep_path": str(directory / "refit" / "prep.joblib"),
            "training_info": {"engine": "tabm_pwl", "inner": inner_info, "refit": refit_info}}


In [11]:
#tambah
def obtain_fold(name, fold):
    outer, valid, learn, stop, _ = inner_split(fold)
    directory = RUN_DIR / "models" / name / f"fold_{fold}"
    directory.mkdir(parents=True, exist_ok=True)
    record_path, prediction_path = directory / "record.json", directory / "predictions.npz"
    signature = hashlib.sha256(json.dumps({"run": RUN_SIGNATURE, "name": name, "fold": fold,
        "outer_train_hash": hash_array(outer), "valid_hash": hash_array(valid),
        "learn_hash": hash_array(learn), "stop_hash": hash_array(stop)}, sort_keys=True).encode()).hexdigest()
    phase_manifest = directory / "job_signature.json"
    if phase_manifest.is_file() and read_json(phase_manifest).get("signature") != signature:
        raise RuntimeError(f"Cache job berbeda: {directory}. Gunakan folder output dengan signature yang sesuai.")
    write_json(phase_manifest, {"signature": signature, "run_signature": RUN_SIGNATURE, "name": name, "fold": fold})
    try:
        if record_path.is_file():
            record = read_json(record_path)
            paths = {key: directory / record[key] for key in ("model_file", "prep_file")}
            cache_valid = (prediction_path.is_file() and record.get("signature") == signature
                    and all(path.is_file() for path in paths.values())
                    and record.get("prediction_sha256") == file_hash(prediction_path)
                    and record.get("model_sha256") == file_hash(paths["model_file"])
                    and record.get("prep_sha256") == file_hash(paths["prep_file"]))
            if not cache_valid:
                raise RuntimeError(f"Cache lengkap gagal pemeriksaan file/hash: {directory}. Pulihkan backup yang utuh sebelum memakai job ini.")
            with np.load(prediction_path, allow_pickle=False) as archive:
                assert np.array_equal(archive["valid_idx"], valid)
                pv, pt = archive["valid"].astype(np.float64), archive["test"].astype(np.float64)
            assert pv.shape == (len(valid),) and pt.shape == (len(TEST_IDS),)
            assert np.isfinite(pv).all() and np.isfinite(pt).all()
            TRAIN_EVENTS.append({"candidate": name, "fold": fold, "status": "cache_reused",
                                 "target_NRMSE": record["target_NRMSE"]})
            log(f"{name} fold {fold}: hasil lengkap dari cache.")
            return {"valid": pv, "test": pt, "record": record}
        if not RUN_TRAINING or not ENABLED[name]:
            reason = "RUN_TRAINING=False" if not RUN_TRAINING else "Dependensi belum tersedia"
            log(f"{name} fold {fold}: dilewati ({reason}).")
            TRAIN_EVENTS.append({"candidate": name, "fold": fold, "status": "skipped", "reason": reason})
            return None
        started = time.monotonic()
        if SPECS[name]["kind"] == "realmlp":
            result = train_realmlp_fold(name, fold, directory)
        else:
            result = train_tabm_fold(name, fold, directory)
        pv, pt = np.asarray(result["valid"], np.float64), np.asarray(result["test"], np.float64)
        if SPECS[name]["transform"] == "divide_by_area":
            pv, pt = pv * AREA[valid], pt * TEST_AREA
        pv, pt = np.maximum(pv, 0.0), np.maximum(pt, 0.0)
        assert pv.shape == (len(valid),) and pt.shape == (len(TEST_IDS),)
        assert np.isfinite(pv).all() and np.isfinite(pt).all()
        model_path, prep_path = Path(result["model_path"]), Path(result["prep_path"])
        target = SPECS[name]["target"]
        record = {"signature": signature, "candidate": name, "fold": fold, "target": TARGETS[target],
            "seconds": time.monotonic() - started,
            "target_NRMSE": float(np.sqrt(np.mean((pv - Y[valid, target]) ** 2)) / SIGMA[target]),
            "model_file": str(model_path.relative_to(directory)), "prep_file": str(prep_path.relative_to(directory)),
            "model_sha256": file_hash(model_path), "prep_sha256": file_hash(prep_path),
            "training_info": result["training_info"], "evaluation": CONFIG["evaluation"]}
        write_npz(prediction_path, valid=pv, test=pt, valid_idx=valid, train_ids=TRAIN_IDS[valid], test_ids=TEST_IDS,
                  target=np.array(TARGETS[target]))
        record["prediction_sha256"] = file_hash(prediction_path)
        write_json(record_path, record)
        write_npz(directory / "split_indices.npz", outer_train=outer, valid=valid, inner_learn=learn, inner_stop=stop)
        TRAIN_EVENTS.append({"candidate": name, "fold": fold, "status": "complete", "seconds": record["seconds"],
                             "target_NRMSE": record["target_NRMSE"]})
        log(f"{name} fold {fold}: selesai {record['seconds']/60:.1f} menit; NRMSE {record['target_NRMSE']:.6f}; model disimpan.")
        return {"valid": pv, "test": pt, "record": record}
    except Exception:
        error = traceback.format_exc()
        ERRORS.append({"candidate": name, "fold": fold, "error": error})
        write_json(RUN_DIR / "errors.json", ERRORS)
        TRAIN_EVENTS.append({"candidate": name, "fold": fold, "status": "failed", "error": error})
        log(error)
        return None
    finally:
        pd.DataFrame(TRAIN_EVENTS).to_csv(RUN_DIR / "reports" / "training_diagnostics.csv", index=False)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def blend_one(reference, vector, area, name, mode, alpha):
    result = reference.copy()
    target = SPECS[name]["target"]
    result[:, target] = (1 - alpha) * reference[:, target] + alpha * vector
    if mode == "eui_gas":
        result[:, 0] = (1 - alpha) * reference[:, 0] + alpha * vector * area
    assert np.isfinite(result).all() and np.array_equal(result[:, 3:], reference[:, 3:])
    return result

def describe_change(name, before, after, indices, folds):
    pooled_before, pooled_after = nrmse(Y[indices], before[indices]), nrmse(Y[indices], after[indices])
    rows = []
    for fold in folds:
        valid = FOLDS[fold][1]
        a, b = nrmse(Y[valid], before[valid]), nrmse(Y[valid], after[valid])
        rows.append({"candidate": name, "fold": fold, "reference_CV": float(a.mean()),
                     "candidate_CV": float(b.mean()), "delta_CV": float(b.mean() - a.mean()),
                     **{f"delta_{TARGETS[t]}": float(b[t] - a[t]) for t in range(5)}})
    deltas = np.array([row["delta_CV"] for row in rows])
    return {"CV": float(pooled_after.mean()), "gain": float(pooled_before.mean() - pooled_after.mean()),
            "NRMSE": pooled_after.tolist(), "improved_folds": int((deltas < -1e-10).sum()),
            "mean_fold_delta": float(deltas.mean()), "std_fold_delta_ddof1": float(deltas.std(ddof=1))}, rows


## Screening dua fold dan penguncian resep

Kelulusan screening memerlukan dua job selesai, gain gabungan minimal 0,00010, dan kedua fold membaik. Untuk EUI, mode EUI-only dan EUI+gas dibandingkan; untuk listrik hanya kolom listrik yang berubah. Alpha dipilih dari daftar tetap pada konfigurasi.

Di antara dua kandidat EUI, pilih satu berdasarkan screening. Jika kandidat listrik juga lolos, kedua komponen digabung dalam resep utama yang dikunci **sebelum** tiga fold konfirmasi. Hasil konfirmasi tidak dipakai untuk mengganti alpha, mode, atau urutan pilihan model.

In [12]:
#tambah
SCREEN_ROWS, SCREEN_DECISIONS = [], {}
SCREEN_IDX = np.concatenate([FOLDS[fold][1] for fold in CONFIG["screen_folds"]])
for name in SPECS:
    for fold in CONFIG["screen_folds"]:
        result = obtain_fold(name, fold)
        if result is not None:
            FOLD_RESULTS[name][fold] = result
    complete = all(fold in FOLD_RESULTS[name] for fold in CONFIG["screen_folds"])
    if not complete:
        SCREEN_DECISIONS[name] = {"passed": False, "reason": "Job screening belum lengkap; periksa dependencies.json dan errors.json",
                                  "complete_folds": sorted(FOLD_RESULTS[name])}
        continue
    modes = ["eui_only", "eui_gas"] if SPECS[name]["target"] == 2 else ["electric"]
    options = []
    for mode in modes:
        for alpha in CONFIG["blend_alphas"]:
            prediction = REF_OOF.copy()
            for fold in CONFIG["screen_folds"]:
                valid = FOLDS[fold][1]
                prediction[valid] = blend_one(REF_OOF[valid], FOLD_RESULTS[name][fold]["valid"], AREA[valid], name, mode, alpha)
            summary, rows = describe_change(name, REF_OOF, prediction, SCREEN_IDX, CONFIG["screen_folds"])
            passed = bool(summary["gain"] >= CONFIG["screen_min_gain"] and summary["improved_folds"] == 2)
            option = {"candidate": name, "mode": mode, "alpha": alpha, **summary, "passed": passed}
            SCREEN_ROWS.append(option)
            options.append(option)
    passing = [option for option in options if option["passed"]]
    best = min(passing if passing else options, key=lambda option: option["CV"])
    SCREEN_DECISIONS[name] = {**best, "complete_folds": CONFIG["screen_folds"],
                              "reason": "Lolos screening" if best["passed"] else "Gain/konsistensi screening belum cukup"}
    log(f"Screen {name}: gain {best['gain']:+.8f}; mode {best['mode']}; alpha {best['alpha']}; lanjut {best['passed']}.")

LOCKED_COMPONENTS = []
for target in (2, 1):
    candidates = [decision for name, decision in SCREEN_DECISIONS.items()
                  if SPECS[name]["target"] == target and decision.get("passed")]
    if candidates:
        choice = min(candidates, key=lambda decision: decision["CV"])
        LOCKED_COMPONENTS.append({"candidate": choice["candidate"], "mode": choice["mode"], "alpha": choice["alpha"],
                                  "screen_gain": choice["gain"]})
LOCKED = {"signature": RUN_SIGNATURE, "screen_folds": CONFIG["screen_folds"],
          "confirm_folds": [fold for fold in range(5) if fold not in CONFIG["screen_folds"]],
          "components": LOCKED_COMPONENTS, "evaluation": CONFIG["evaluation"],
          "rule": "no confirmation-based retuning or replacement of locked components"}
write_json(RUN_DIR / "screening.json", SCREEN_DECISIONS)
write_json(RUN_DIR / "parameters" / "locked_recipe.json", LOCKED)
pd.DataFrame(SCREEN_ROWS).to_csv(RUN_DIR / "reports" / "screening.csv", index=False)
display(pd.DataFrame([{"candidate": name, **decision} for name, decision in SCREEN_DECISIONS.items()]))
log(f"Komponen terkunci: {LOCKED_COMPONENTS}")


[19:16:01] realmlp_eui fold 0: membangun preprocessing inner-training.
[19:16:06] realmlp_eui fold 0: seleksi epoch inner-subzone, lalu native refit seluruh 343,442 outer-train.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Columns classified as continuous: ['bld_dim_01', 'bld_dim_02', 'socio_demog_002', 'bld_dim_03', 'bld_dim_04', 'bld_dim_05', 'bld_dim_06', 'bld_dim_07', 'bld_dim_08', 'bld_dim_09', 'bld_dim_10', 'bld_dim_11', 'bld_dim_12', 'bld_dim_13', 'socio_demog_003', 'bld_dim_14', 'bld_dim_15', 'bld_dim_16', 'bld_dim_17', 'bld_era_02', 'bld_era_03', 'bld_era_04', 'bld_era_05', 'bld_era_06', 'bld_era_07', 'bld_era_08', 'bld_era_09', 'bld_era_10', 'bld_era_11', 'bld_era_12', 'bld_era_13', 'bld_era_14', 'bld_era_15', 'socio_demog_005', 'bld_dim_19', 'socio_demog_006', 'bld_dim_20', 'bld_dim_21', 'bld_dim_22', 'bld_dim_23', 'bld_dim_24', 'bld_dim_25', 'bld_dim_26', 'bld_dim_27', 'bld_dim_28', 'bld_dim_29', 'bld_dim_30', 'socio_demog_007', 'socio_demog_008', 'socio_demog_009', 'socio_demog_010', 'bld_dim_31', 'bld_dim_32', 'socio_demog_011', 'bld_dim_33', 'bld_dim_34', 'socio_demog_012', 'bld_dim_35', 'socio_demog_013', 'bld_dim_36', 'socio_demog_014', 'bld_dim_37', 'bld_dim_38', 'bld_dim_39', 'bld_dim_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 1/256: val rmse = 29.096498
Epoch 2/256: val rmse = 29.124269
Epoch 3/256: val rmse = 29.081373
Epoch 4/256: val rmse = 28.035648
Epoch 5/256: val rmse = 29.447735
Epoch 6/256: val rmse = 28.572042
Epoch 7/256: val rmse = 28.742691
Epoch 8/256: val rmse = 29.751680
[19:16:36] realmlp_eui fold 0 native training/refit: masih berjalan, 0.5 menit.
Epoch 9/256: val rmse = 28.573936
Epoch 10/256: val rmse = 28.391947
Epoch 11/256: val rmse = 28.566584
Epoch 12/256: val rmse = 27.975121
Epoch 13/256: val rmse = 27.943825
Epoch 14/256: val rmse = 27.458462
Epoch 15/256: val rmse = 27.188911
Epoch 16/256: val rmse = 27.014565
Epoch 17/256: val rmse = 27.016773
Epoch 18/256: val rmse = 27.087502
Epoch 19/256: val rmse = 27.333368
[19:17:06] realmlp_eui fold 0 native training/refit: masih berjalan, 1.0 menit.
Epoch 20/256: val rmse = 27.623238
Epoch 21/256: val rmse = 27.758326
Epoch 22/256: val rmse = 27.698713
Epoch 23/256: val rmse = 27.835302
Epoch 24/256: val rmse = 28.378983
Epoch 25/

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


... (45 outputs left out)

/usr/local/lib/python3.13/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 11-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 34-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rtdl_num_embeddings.py:340: UserWarning: The 38-th feature has just two bin edges, which means only one bin. Strictly speaking, using a single bin for the piecewise-linear encoding should not break anything, but it is the same as using sklearn.preprocessing.MinMaxScaler
  warnings.warn(
/usr/local/lib/p

[19:42:47] tabm_pwl_eui fold 1: refit 15 epoch TabM-PWL: masih berjalan, 0.5 menit.
[19:42:57] TabM fold 1 refit: epoch 1/15, MSE 0.519260, 11.3 detik.
[19:43:08] TabM fold 1 refit: epoch 2/15, MSE 0.485796, 11.3 detik.
[19:43:17] tabm_pwl_eui fold 1: refit 15 epoch TabM-PWL: masih berjalan, 1.0 menit.
[19:43:20] TabM fold 1 refit: epoch 3/15, MSE 0.476351, 11.4 detik.
[19:43:31] TabM fold 1 refit: epoch 4/15, MSE 0.468707, 11.4 detik.
[19:43:43] TabM fold 1 refit: epoch 5/15, MSE 0.461829, 11.4 detik.
[19:43:47] tabm_pwl_eui fold 1: refit 15 epoch TabM-PWL: masih berjalan, 1.5 menit.
[19:43:54] TabM fold 1 refit: epoch 6/15, MSE 0.457509, 11.4 detik.
[19:44:06] TabM fold 1 refit: epoch 7/15, MSE 0.452770, 11.3 detik.
[19:44:17] TabM fold 1 refit: epoch 8/15, MSE 0.447433, 11.3 detik.
[19:44:17] tabm_pwl_eui fold 1: refit 15 epoch TabM-PWL: masih berjalan, 2.0 menit.
[19:44:28] TabM fold 1 refit: epoch 9/15, MSE 0.442945, 11.3 detik.
[19:44:40] TabM fold 1 refit: epoch 10/15, MSE 0.438

,candidate,mode,alpha,CV,gain,NRMSE,improved_folds,mean_fold_delta,std_fold_delta_ddof1,passed,complete_folds,reason
0,realmlp_eui,eui_gas,0.25,0.268452,0.000312,"[0.24864356261543058, 0.317005986675429, 0.658...",2,-0.000312,0.000008,True,"[0, 1]",Lolos screening
1,realmlp_electric_area,electric,0.25,0.268649,0.000115,"[0.2490878748262549, 0.3164301507427673, 0.659...",2,-0.000115,0.000021,True,"[0, 1]",Lolos screening
2,tabm_pwl_eui,eui_gas,0.25,0.268551,0.000213,"[0.24887600562207257, 0.317005986675429, 0.658...",2,-0.000213,0.000045,True,"[0, 1]",Lolos screening


[19:45:44] Komponen terkunci: [{'candidate': 'realmlp_eui', 'mode': 'eui_gas', 'alpha': 0.25, 'screen_gain': 0.00031215441683896117}, {'candidate': 'realmlp_electric_area', 'mode': 'electric', 'alpha': 0.25, 'screen_gain': 0.00011516718653237223}]


In [13]:
#tambah
CONFIRM_FOLDS = LOCKED["confirm_folds"]
if AUTO_COMPLETE and LOCKED_COMPONENTS:
    for component in LOCKED_COMPONENTS:
        name = component["candidate"]
        for fold in CONFIRM_FOLDS:
            result = obtain_fold(name, fold)
            if result is not None:
                FOLD_RESULTS[name][fold] = result
else:
    log("Fold konfirmasi dilewati: tidak ada komponen lolos atau AUTO_COMPLETE=False.")
FULL_MODELS = {}
MODEL_COVERAGE = []
for name, results in FOLD_RESULTS.items():
    target = SPECS[name]["target"]
    complete_folds = sorted(results)
    count = sum(len(FOLDS[fold][1]) for fold in complete_folds)
    sse = sum(float(np.sum((results[fold]["valid"] - Y[FOLDS[fold][1], target]) ** 2)) for fold in complete_folds)
    rmse = float(np.sqrt(sse / count)) if count else None
    score = rmse / float(SIGMA[target]) if count else None
    MODEL_COVERAGE.append({"candidate": name, "target": TARGETS[target], "completed_folds": len(results),
        "scored_rows": count, "available_folds": ",".join(map(str, complete_folds)),
        "available_folds_target_RMSE": rmse, "available_folds_target_NRMSE": score,
        "five_fold_target_NRMSE": score if len(results) == 5 else None,
        "CV_status": "complete_5_fold" if len(results) == 5 else "partial" if count else "no_predictions"})
    if set(results) != set(range(5)):
        log(f"{name}: fold lengkap {len(results)}/5; full CV dan test ensemble belum tersedia.")
        continue
    vector = np.full(len(Y), np.nan, dtype=np.float64)
    test_vector = np.zeros(len(TEST_IDS), dtype=np.float64)
    for fold in range(5):
        valid = FOLDS[fold][1]
        vector[valid] = results[fold]["valid"]
        test_vector += results[fold]["test"] / 5.0
    assert np.isfinite(vector).all() and np.isfinite(test_vector).all()
    FULL_MODELS[name] = {"oof": vector, "test": test_vector}
    write_npz(RUN_DIR / "predictions" / f"model_{name}.npz", oof=vector, test=test_vector,
              train_ids=TRAIN_IDS, test_ids=TEST_IDS, fold_id=FOLD_ID, target=np.array(TARGETS[SPECS[name]["target"]]))
write_json(RUN_DIR / "errors.json", ERRORS)
pd.DataFrame(MODEL_COVERAGE).to_csv(RUN_DIR / "reports" / "model_coverage.csv", index=False)
display(pd.DataFrame(MODEL_COVERAGE))


[19:45:46] realmlp_eui fold 2: membangun preprocessing inner-training.
[19:45:50] realmlp_eui fold 2: seleksi epoch inner-subzone, lalu native refit seluruh 343,442 outer-train.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Columns classified as continuous: ['bld_dim_01', 'bld_dim_02', 'socio_demog_002', 'bld_dim_03', 'bld_dim_04', 'bld_dim_05', 'bld_dim_06', 'bld_dim_07', 'bld_dim_08', 'bld_dim_09', 'bld_dim_10', 'bld_dim_11', 'bld_dim_12', 'bld_dim_13', 'socio_demog_003', 'bld_dim_14', 'bld_dim_15', 'bld_dim_16', 'bld_dim_17', 'bld_era_02', 'bld_era_03', 'bld_era_04', 'bld_era_05', 'bld_era_06', 'bld_era_07', 'bld_era_08', 'bld_era_09', 'bld_era_10', 'bld_era_11', 'bld_era_12', 'bld_era_13', 'bld_era_14', 'bld_era_15', 'socio_demog_005', 'bld_dim_19', 'socio_demog_006', 'bld_dim_20', 'bld_dim_21', 'bld_dim_22', 'bld_dim_23', 'bld_dim_24', 'bld_dim_25', 'bld_dim_26', 'bld_dim_27', 'bld_dim_28', 'bld_dim_29', 'bld_dim_30', 'socio_demog_007', 'socio_demog_008', 'socio_demog_009', 'socio_demog_010', 'bld_dim_31', 'bld_dim_32', 'socio_demog_011', 'bld_dim_33', 'bld_dim_34', 'socio_demog_012', 'bld_dim_35', 'socio_demog_013', 'bld_dim_36', 'socio_demog_014', 'bld_dim_37', 'bld_dim_38', 'bld_dim_39', 'bld_dim_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 1/256: val rmse = 29.902290
Epoch 2/256: val rmse = 29.015623
Epoch 3/256: val rmse = 29.079069
Epoch 4/256: val rmse = 28.632765
Epoch 5/256: val rmse = 29.258766
Epoch 6/256: val rmse = 29.450026
Epoch 7/256: val rmse = 29.217243
Epoch 8/256: val rmse = 29.177765
Epoch 9/256: val rmse = 28.680082
[19:46:20] realmlp_eui fold 2 native training/refit: masih berjalan, 0.5 menit.
Epoch 10/256: val rmse = 29.234705
Epoch 13/256: val rmse = 28.292555
Epoch 14/256: val rmse = 27.894751
Epoch 15/256: val rmse = 27.729275
Epoch 16/256: val rmse = 27.478392
Epoch 17/256: val rmse = 27.478018
Epoch 18/256: val rmse = 27.560305
Epoch 19/256: val rmse = 27.775927
[19:46:50] realmlp_eui fold 2 native training/refit: masih berjalan, 1.0 menit.
Epoch 20/256: val rmse = 27.914633
Epoch 21/256: val rmse = 28.536406
Epoch 22/256: val rmse = 28.246687
Epoch 23/256: val rmse = 28.402811
Epoch 24/256: val rmse = 29.150280
Epoch 25/256: val rmse = 28.629484
Epoch 26/256: val rmse = 29.351944
Epoch 27/

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


... (52 outputs left out)

Refit Epoch 1/17
Refit Epoch 2/17
Refit Epoch 3/17
Refit Epoch 4/17
[20:01:51] realmlp_electric_area fold 4 native training/refit: masih berjalan, 2.0 menit.
Refit Epoch 5/17
Refit Epoch 6/17
Refit Epoch 7/17
Refit Epoch 8/17
Refit Epoch 9/17
Refit Epoch 10/17
Refit Epoch 11/17
Refit Epoch 12/17
Refit Epoch 13/17
[20:02:21] realmlp_electric_area fold 4 native training/refit: masih berjalan, 2.5 menit.
Refit Epoch 14/17
Refit Epoch 15/17
Refit Epoch 16/17
Refit Epoch 17/17
[20:02:33] realmlp_electric_area fold 4: menyimpan estimator resmi.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: Fa

[20:02:38] realmlp_electric_area fold 4: selesai 2.9 menit; NRMSE 0.325012; model disimpan.
[20:02:40] tabm_pwl_eui: fold lengkap 2/5; full CV dan test ensemble belum tersedia.


,candidate,target,completed_folds,scored_rows,available_folds,available_folds_target_RMSE,available_folds_target_NRMSE,five_fold_target_NRMSE,CV_status
0,realmlp_eui,target_eui_gas_kwh_per_sqm,5,429303,"0,1,2,3,4",27.215389,0.669505,0.669505,complete_5_fold
1,realmlp_electric_area,target_macro_elec_kwh,5,429303,"0,1,2,3,4",13473.748937,0.322915,0.322915,complete_5_fold
2,tabm_pwl_eui,target_eui_gas_kwh_per_sqm,2,171722,"0,1",27.105727,0.666808,NaN,partial


## Konfirmasi, laporan per kolom, dan submission

Resep utama diterima bila seluruh komponen lengkap lima fold, gain pada tiga fold konfirmasi minimal 0,00010 dengan **3/3** fold membaik, serta gain full OOF minimal 0,00010 dengan minimal **4/5** fold membaik. Full OOF bukan set konfirmasi tambahan.

Komponen lengkap yang membaik juga dapat diekspor sebagai `DIAGNOSTIC`, tanpa mengganti pilihan utama yang sudah dikunci. Jika resep utama gagal, `main=null`; tidak ada CSV utama yang mengulang v17.

**Listrik/luas:** RealMLP resmi tidak menyediakan sample weights. Kandidat ini mengoptimalkan MSE intensitas tanpa bobot; itu tidak sama dengan MSE total listrik (yang memerlukan bobot luas²). Semua keputusan kelulusan memakai prediksi total kWh setelah dikalikan luas.

In [14]:
#tambah
METRIC_ROWS, COLUMN_ROWS, FOLD_ROWS, SUBMISSIONS = [], [], [], []
PREDICTIONS = {"reference_v17": {"oof": REF_OOF, "test": REF_NOLOOKUP, "test_lookup": REF_TEST}}
for component in LOCKED_COMPONENTS:
    name = component["candidate"]
    if name in FULL_MODELS:
        model = FULL_MODELS[name]
        oof = blend_one(REF_OOF, model["oof"], AREA, name, component["mode"], component["alpha"])
        test_values = blend_one(REF_NOLOOKUP, model["test"], TEST_AREA, name, component["mode"], component["alpha"])
        lookup_values = test_values.copy()
        lookup_values[:, 4] = S28_TEST[:, 4]
        PREDICTIONS[name] = {"oof": oof, "test": test_values, "test_lookup": lookup_values}

LOCKED_COMPLETE = bool(LOCKED_COMPONENTS and all(component["candidate"] in FULL_MODELS for component in LOCKED_COMPONENTS))
if LOCKED_COMPLETE:
    oof, test_values = REF_OOF.copy(), REF_NOLOOKUP.copy()
    for component in LOCKED_COMPONENTS:
        name = component["candidate"]
        model = FULL_MODELS[name]
        oof = blend_one(oof, model["oof"], AREA, name, component["mode"], component["alpha"])
        test_values = blend_one(test_values, model["test"], TEST_AREA, name, component["mode"], component["alpha"])
    lookup_values = test_values.copy()
    lookup_values[:, 4] = S28_TEST[:, 4]
    PREDICTIONS["locked_recipe"] = {"oof": oof, "test": test_values, "test_lookup": lookup_values}

CONFIRM_IDX = np.concatenate([FOLDS[fold][1] for fold in CONFIRM_FOLDS])
SUMMARIES = {}
for name, predictions in PREDICTIONS.items():
    full, rows = describe_change(name, REF_OOF, predictions["oof"], np.arange(len(Y)), list(range(5)))
    confirm, _ = describe_change(name, REF_OOF, predictions["oof"], CONFIRM_IDX, CONFIRM_FOLDS)
    SUMMARIES[name] = {"full": full, "confirm": confirm}
    FOLD_ROWS.extend(rows)
    METRIC_ROWS.append({"candidate": name, **dict(zip(TARGETS, full["NRMSE"])),
        "CV": full["CV"], "gain_vs_v17": full["gain"], "improved_folds": full["improved_folds"],
        "mean_fold_delta": full["mean_fold_delta"], "std_fold_delta_ddof1": full["std_fold_delta_ddof1"],
        "confirmation_CV": confirm["CV"], "confirmation_gain": confirm["gain"],
        "confirmation_improved_folds": confirm["improved_folds"], "current_public": None,
        "evaluation": CONFIG["evaluation"]})
    for target in range(5):
        COLUMN_ROWS.append({"candidate": name, "target": TARGETS[target], "RMSE": full["NRMSE"][target] * float(SIGMA[target]),
            "NRMSE": full["NRMSE"][target], "normalizer": float(SIGMA[target]),
            "reference_NRMSE": float(REF_SCORES[target]), "delta_NRMSE": full["NRMSE"][target] - float(REF_SCORES[target])})
    if name != "reference_v17":
        write_npz(RUN_DIR / "predictions" / f"predictions_{name}.npz", oof=predictions["oof"], test=predictions["test"],
                  test_lookup=predictions["test_lookup"], train_ids=TRAIN_IDS, test_ids=TEST_IDS,
                  fold_id=FOLD_ID, target_names=np.array(TARGETS))

NEW_SELECTED = False
if LOCKED_COMPLETE:
    full, confirm = SUMMARIES["locked_recipe"]["full"], SUMMARIES["locked_recipe"]["confirm"]
    changed = not np.allclose(PREDICTIONS["locked_recipe"]["test"][:, :3], REF_NOLOOKUP[:, :3], rtol=1e-12, atol=1e-9)
    NEW_SELECTED = bool(changed and full["gain"] >= CONFIG["full_min_gain"]
        and full["improved_folds"] >= CONFIG["full_min_improved_folds"]
        and confirm["gain"] >= CONFIG["confirm_min_gain"] and confirm["improved_folds"] == 3)

def export_csv(path, predictions):
    assert predictions.shape == (len(TEST_IDS), 5) and np.isfinite(predictions).all()
    assert (predictions[:, :3] >= 0).all() and ((predictions[:, 3:] >= 0) & (predictions[:, 3:] <= 100)).all()
    assert np.array_equal(predictions[:, 3], REF_NOLOOKUP[:, 3])
    frame = SAMPLE.copy()
    frame[TARGETS] = predictions
    frame.to_csv(path, index=False, float_format="%.17g")
    return file_hash(path)

MAIN_PATH = None
seen = {hash_array(REF_TEST)}
export_order = (["locked_recipe"] if NEW_SELECTED else []) + [name for name in PREDICTIONS if name != "reference_v17"]
for name in dict.fromkeys(export_order):
    result = PREDICTIONS[name]
    if SUMMARIES[name]["full"]["gain"] <= 1e-10:
        continue
    digest = hash_array(result["test_lookup"])
    if digest in seen:
        continue
    seen.add(digest)
    role = "selected" if name == "locked_recipe" and NEW_SELECTED else "DIAGNOSTIC"
    path = RUN_DIR / "submissions" / f"submission_v21_{role}_{name}.csv"
    sha = export_csv(path, result["test_lookup"])
    no_lookup_path = path.with_name(path.stem + "_nolookup.csv")
    export_csv(no_lookup_path, result["test"])
    SUBMISSIONS.append({"candidate": name, "role": role, "path": str(path), "nolookup_path": str(no_lookup_path),
        "sha256": sha, "CV": SUMMARIES[name]["full"]["CV"], "confirmation_gain": SUMMARIES[name]["confirm"]["gain"],
        "current_public": None})
    if role == "selected":
        MAIN_PATH = path

STATUS = ("new_candidate_selected" if NEW_SELECTED else "confirmation_rejected" if LOCKED_COMPLETE
          else "confirmation_incomplete" if LOCKED_COMPONENTS else "screen_rejected_or_incomplete")
SELECTION = {"signature": RUN_SIGNATURE, "status": STATUS, "new_candidate_selected": NEW_SELECTED,
    "main": str(MAIN_PATH) if MAIN_PATH else None, "locked_components": LOCKED_COMPONENTS,
    "complete_folds": {name: len(results) for name, results in FOLD_RESULTS.items()},
    "reference_CV": REF_CV, "selected_CV": SUMMARIES["locked_recipe"]["full"]["CV"] if NEW_SELECTED else None,
    "locked_recipe_summary": SUMMARIES.get("locked_recipe"), "current_public": None,
    "reference_public_historical_reported": REFERENCE_PUBLIC,
    "evaluation": CONFIG["evaluation"], "old_models_retrained": 0}
write_json(RUN_DIR / "selection.json", SELECTION)
pd.DataFrame(METRIC_ROWS).to_csv(RUN_DIR / "reports" / "oof_cv_summary.csv", index=False)
pd.DataFrame(COLUMN_ROWS).to_csv(RUN_DIR / "reports" / "column_wise_rmse_nrmse.csv", index=False)
pd.DataFrame(FOLD_ROWS).to_csv(RUN_DIR / "reports" / "oof_fold_comparison.csv", index=False)
pd.DataFrame(SUBMISSIONS, columns=["candidate", "role", "path", "nolookup_path", "sha256", "CV", "confirmation_gain", "current_public"]).to_csv(
    RUN_DIR / "submission_catalog.csv", index=False)
with pd.option_context("display.max_columns", None, "display.max_colwidth", 90):
    display(pd.DataFrame(METRIC_ROWS))
    display(pd.DataFrame(COLUMN_ROWS))
    if "locked_recipe" in SUMMARIES:
        display(pd.DataFrame(FOLD_ROWS).query("candidate == 'locked_recipe'"))
log(json.dumps(SELECTION, indent=2, default=json_value))
if MAIN_PATH:
    display(FileLink(str(MAIN_PATH)))
else:
    log("Tidak ada submission utama baru. Pertahankan acuan lama; lihat alasan screening/konfirmasi.")
if SUBMISSIONS:
    display(pd.DataFrame(SUBMISSIONS))


,candidate,target_macro_gas_kwh,target_macro_elec_kwh,target_eui_gas_kwh_per_sqm,target_clean_building_ratio,target_electric_heating_adoption,CV,gain_vs_v17,improved_folds,mean_fold_delta,std_fold_delta_ddof1,confirmation_CV,confirmation_gain,confirmation_improved_folds,current_public,evaluation
0,reference_v17,0.250231,0.319368,0.660092,1.066132e-07,0.117813,0.269501,0.000000,0,0.000000,0.000000,0.269989,0.000000,0,None,new_neural_group_oof_with_cached_S28_v17_reference_exploratory
1,realmlp_eui,0.249892,0.319368,0.658940,1.066132e-07,0.117813,0.269202,0.000298,5,-0.000298,0.000013,0.269700,0.000289,3,None,new_neural_group_oof_with_cached_S28_v17_reference_exploratory
2,realmlp_electric_area,0.250231,0.318912,0.660092,1.066132e-07,0.117813,0.269410,0.000091,5,-0.000091,0.000027,0.269914,0.000075,3,None,new_neural_group_oof_with_cached_S28_v17_reference_exploratory
3,locked_recipe,0.249892,0.318912,0.658940,1.066132e-07,0.117813,0.269111,0.000389,5,-0.000390,0.000040,0.269624,0.000365,3,None,new_neural_group_oof_with_cached_S28_v17_reference_exploratory


,candidate,target,RMSE,NRMSE,normalizer,reference_NRMSE,delta_NRMSE
0,reference_v17,target_macro_gas_kwh,37972.587579,2.502312e-01,151750.04,2.502312e-01,0.000000
1,reference_v17,target_macro_elec_kwh,13325.729144,3.193675e-01,41725.37,3.193675e-01,0.000000
2,reference_v17,target_eui_gas_kwh_per_sqm,26.832733,6.600918e-01,40.65,6.600918e-01,0.000000
3,reference_v17,target_clean_building_ratio,0.000001,1.066132e-07,12.49,1.066132e-07,0.000000
4,reference_v17,target_electric_heating_adoption,0.845900,1.178133e-01,7.18,1.178133e-01,0.000000
5,realmlp_eui,target_macro_gas_kwh,37921.053465,2.498916e-01,151750.04,2.502312e-01,-0.000340
6,realmlp_eui,target_macro_elec_kwh,13325.729144,3.193675e-01,41725.37,3.193675e-01,0.000000
7,realmlp_eui,target_eui_gas_kwh_per_sqm,26.785896,6.589396e-01,40.65,6.600918e-01,-0.001152
8,realmlp_eui,target_clean_building_ratio,0.000001,1.066132e-07,12.49,1.066132e-07,0.000000
9,realmlp_eui,target_electric_heating_adoption,0.845900,1.178133e-01,7.18,1.178133e-01,0.000000


,candidate,fold,reference_CV,candidate_CV,delta_CV,delta_target_macro_gas_kwh,delta_target_macro_elec_kwh,delta_target_eui_gas_kwh_per_sqm,delta_target_clean_building_ratio,delta_target_electric_heating_adoption
15,locked_recipe,0,0.267867,0.267460,-0.000407,-0.000497,-0.000502,-0.001036,0.0,0.0
16,locked_recipe,1,0.269658,0.269210,-0.000447,-0.000391,-0.000649,-0.001197,0.0,0.0
17,locked_recipe,2,0.269751,0.269389,-0.000362,-0.000224,-0.000375,-0.001212,0.0,0.0
18,locked_recipe,3,0.269764,0.269418,-0.000345,-0.000366,-0.000291,-0.001070,0.0,0.0
19,locked_recipe,4,0.270451,0.270064,-0.000386,-0.000222,-0.000464,-0.001246,0.0,0.0


[20:03:00] {
  "signature": "926c7d3537eabbb2284382c15c8681f69cbfa80a060daf5dcbfac4d0b156327e",
  "status": "new_candidate_selected",
  "new_candidate_selected": true,
  "main": "/kaggle/working/outputs/dicefest_v21/926c7d3537eabbb2/submissions/submission_v21_selected_locked_recipe.csv",
  "locked_components": [
    {
      "candidate": "realmlp_eui",
      "mode": "eui_gas",
      "alpha": 0.25,
      "screen_gain": 0.00031215441683896117
    },
    {
      "candidate": "realmlp_electric_area",
      "mode": "electric",
      "alpha": 0.25,
      "screen_gain": 0.00011516718653237223
    }
  ],
  "complete_folds": {
    "realmlp_eui": 5,
    "realmlp_electric_area": 5,
    "tabm_pwl_eui": 2
  },
  "reference_CV": 0.26950079729861065,
  "selected_CV": 0.26911130819292717,
  "locked_recipe_summary": {
    "full": {
      "CV": 0.26911130819292717,
      "gain": 0.0003894891056834826,
      "NRMSE": [
        0.24989155498906984,
        0.31891190038883566,
        0.6589396398032945,
 

/kaggle/working/outputs/dicefest_v21/926c7d3537eabbb2/submissions/submission_v21_selected_locked_recipe.csv

,candidate,role,path,nolookup_path,sha256,CV,confirmation_gain,current_public
0,locked_recipe,selected,/kaggle/working/outputs/dicefest_v21/926c7d353...,/kaggle/working/outputs/dicefest_v21/926c7d353...,f1907c3c2edc52b141b8975d24ebb45cbbbc7fd5dcb6f6...,0.269111,0.000365,None
1,realmlp_eui,DIAGNOSTIC,/kaggle/working/outputs/dicefest_v21/926c7d353...,/kaggle/working/outputs/dicefest_v21/926c7d353...,af5c1bc80584efa990247b5e0852b750a3d12724b23287...,0.269202,0.000289,None
2,realmlp_electric_area,DIAGNOSTIC,/kaggle/working/outputs/dicefest_v21/926c7d353...,/kaggle/working/outputs/dicefest_v21/926c7d353...,f189047ca77e4a1d4664e5d52ef138a25050d28d58ed99...,0.269410,0.000075,None


## Backup dan pemakaian ulang

Semua model/preprocessor per fold, checkpoint TabM, prediksi, split indices, parameter, error, dan laporan disimpan. ZIP juga membawa NPZ Dani beserta metadata; CSV kompetisi tidak digandakan. Untuk memulihkan run, ekstrak ZIP sebagai Input Kaggle dengan versi dependensi/config yang sama. Notebook mencari `reproduction_v21.json` dengan signature cocok.

Model RealMLP disimpan sebagai estimator resmi `joblib`; saat memuat, panggil `model.to("cpu")` atau `model.to("cuda:0")` tanpa menugaskan nilai baliknya. Preprocessor eksternal dan manifest fitur diperlukan untuk merekonstruksi input. TabM menyimpan bobot, arsitektur/PWL bins, dan preprocessing setiap tahap.

Checkpoint bukan pengganti notebook hasil run: unduh ZIP **dan** notebook yang memiliki output sebelum sesi berakhir.

In [15]:
#tambah
for source in (SOURCE_PATH, META_PATH):
    shutil.copy2(source, RUN_DIR / "inputs" / source.name)
write_json(RUN_DIR / "reproduction_v21.json", {
    "signature": RUN_SIGNATURE, "config": CONFIG, "source_sha256": SOURCE_SHA,
    "data_hashes": DATA_HASHES, "target_order": TARGETS, "sigma": SIGMA.tolist(),
    "fold_group": "subzone_id", "fold_hash": hash_array(FOLD_ID), "reference_cv": REF_CV,
    "versions": VERSIONS, "status": STATUS, "selection": SELECTION,
    "lookup_coverage": float(LOOKUP_MASK.mean()), "seconds": time.monotonic() - STARTED,
    "evaluation": CONFIG["evaluation"], "old_models_retrained": 0,
})
inventory = [{"path": str(path.relative_to(RUN_DIR)), "bytes": path.stat().st_size, "sha256": file_hash(path)}
             for path in sorted((RUN_DIR / "models").rglob("*")) if path.is_file() and path.suffix != ".tmp"]
pd.DataFrame(inventory, columns=["path", "bytes", "sha256"]).to_csv(RUN_DIR / "model_inventory.csv", index=False)
if RUN_BACKUP:
    stamp = datetime.now(ZoneInfo("Asia/Jakarta"))
    backup_path = WORK / f"dicefest_v21_backup_{stamp:%Y%m%d_%H%M%S_%f}.zip"
    files = [path for path in sorted(RUN_DIR.rglob("*")) if path.is_file() and path.suffix not in (".tmp", ".partial", ".zip")]
    size = sum(path.stat().st_size for path in files)
    if shutil.disk_usage(WORK).free < size + 100 * 1024 ** 2:
        log("Ruang disk tidak cukup untuk ZIP tambahan; folder output tetap tersimpan.")
    else:
        temporary = backup_path.with_name(backup_path.name + ".partial")
        log(f"Membuat backup {len(files)} file, sekitar {size / 1024 ** 3:.3f} GiB.")
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for number, path in enumerate(files, 1):
                archive.write(path, arcname=f"outputs/dicefest_v21/{RUN_SIGNATURE[:16]}/{path.relative_to(RUN_DIR).as_posix()}")
                if number % 20 == 0:
                    log(f"Backup {number}/{len(files)} file.")
            archive.writestr("backup_manifest.json", json.dumps({"signature": RUN_SIGNATURE, "created_at": stamp.isoformat(),
                "selection": SELECTION, "file_count": len(files) + 1,
                "files": [{"path": str(path.relative_to(RUN_DIR)), "bytes": path.stat().st_size} for path in files]},
                indent=2, default=json_value))
        temporary.replace(backup_path)
        log(f"ZIP selesai: {backup_path}; {backup_path.stat().st_size / 1024 ** 3:.3f} GiB.")
        display(FileLink(str(backup_path)))
log(f"SELESAI: {STATUS}; {(time.monotonic() - STARTED) / 60:.1f} menit.")
log("Unduh ZIP dan notebook hasil run sebelum menutup sesi Kaggle.")


[20:03:01] Membuat backup 124 file, sekitar 0.414 GiB.
[20:03:01] Backup 20/124 file.
[20:03:01] Backup 40/124 file.
[20:03:01] Backup 60/124 file.
[20:03:01] Backup 80/124 file.
[20:03:01] Backup 100/124 file.
[20:03:02] Backup 120/124 file.
[20:03:02] ZIP selesai: /kaggle/working/dicefest_v21_backup_20261019_200301_561256.zip; 0.414 GiB.


/kaggle/working/dicefest_v21_backup_20261019_200301_561256.zip

[20:03:02] SELESAI: new_candidate_selected; 48.7 menit.
[20:03:02] Unduh ZIP dan notebook hasil run sebelum menutup sesi Kaggle.


## Sumber dan membaca hasil

- [RealMLP resmi / PyTabKit](https://github.com/dholzmueller/pytabkit), pin `1.7.3`.
- [TabM resmi](https://github.com/yandex-research/tabm), pin `0.0.3`.
- [Numerical embeddings resmi](https://github.com/yandex-research/rtdl-num-embeddings), pin `0.0.12`.
- [Contoh solusi Kaggle RealMLP + TabM](https://www.kaggle.com/competitions/prediction-interval-competition-ii-house-price/discussion/591534); tugas dan metrik berbeda dari DICE.

Lihat `selection.json` untuk keputusan utama. `reports/screening.csv` berisi seluruh opsi screening; `parameters/locked_recipe.json` menunjukkan resep yang dikunci sebelum konfirmasi. `reports/model_coverage.csv` memuat RMSE/NRMSE target model mentah serta jumlah fold yang tersedia; metrik parsial diberi label dan tidak disamakan dengan lima fold. `reports/oof_cv_summary.csv` memisahkan full CV dan tiga fold konfirmasi. `reports/column_wise_rmse_nrmse.csv` dan `oof_fold_comparison.csv` memuat hasil per target dan per fold. `errors.json` serta `dependencies.json` membedakan kegagalan implementasi/dependensi dari kandidat yang memang lemah.

Jika tidak ada model lengkap lima fold, CV kandidat tersebut **belum diketahui**. Notebook tidak mengisi fold hilang dengan acuan atau merata-ratakan test dari jumlah fold yang tidak lengkap.